In [97]:
import logging
import copy
import os
from typing import Dict, Tuple, List, Union

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import ScalarFormatter
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import rcParams
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import numpy as np
from scipy.optimize import (
    dual_annealing,
    minimize as scipy_minimize,
    root,
    least_squares,
)
from joblib import Parallel, delayed
from tqdm import tqdm
import casadi as ca
import math
from itertools import product
import pint


# initialize pint unit registry
ureg = pint.UnitRegistry()

ureg.formatter.default_format = "~P"  # Pretty formatting (uses µ instead of u)

In [98]:
# Ensure the directory for plots exists
if not os.path.exists("plots"):
    os.makedirs("plots")

In [99]:
# Configure logging
logging.basicConfig(
    filename="optimization_log.txt",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

In [100]:
E_mes_BOUND = (0, 1e-3)  # kg/s

F_BOUND = (0, 1e-2)  # kg/s
temp_lower_bound = 250  # Kelvin
temp_upper_bound = 350  # K

# Updated bounds for E_mes1, E_mes2, and F
bounds = [E_mes_BOUND, F_BOUND]

In [101]:
R = 8.314 * ureg.J / (ureg.mol * ureg.K)  # Universal gas constant in J/(mol*K)
T = 298.15 * ureg.K  # Standard temperature in Kelvin
P = 20 * ureg.Pa

C = P / (R * T)  # Concentration in mol/m^3
C.to_base_units()  # Convert to mol/m^3

<Quantity(0.008068358033871856, 'mole / meter ** 3')>

In [102]:
class Param_value:
    def __init__(self, value: float, unit: str):
        # Initialize with a value and unit within pint's unit registry

        # If no unit is passed, or if None is passed, set to dimensionless
        if unit is None or unit == "":
            unit = "dimensionless"
        if unit == "degC":
            # Convert Celsius to Kelvin
            value += 273.15
            unit = "degK"
        if not isinstance(value, (int, float, np.ndarray)):
            raise TypeError(
                f"Value must be a number or an array, got {type(value)} instead."
            )

        self.quantity = value * ureg(unit)

    def __repr__(self):
        # Return a string representation of the quantity with its unit
        return f"{self.quantity}"

    def __str__(self):
        # Return a string representation of the quantity with its unit
        return f"{self.quantity}"

    @property
    def value(self):
        # returns the value, without the unit
        if isinstance(self, tuple):
            return tuple(v.quantity.magnitude for v in self)
        return self.quantity.magnitude

    @property
    def unit(self):
        # return the unit without the value
        if isinstance(self, tuple):
            # return the unit of the first element in the tuple
            return str(self[0].quantity.units)

        return str(self.quantity.units)

    @classmethod
    def set_from_dict(cls, value_key: str, param_dict: Dict[str, Union[float, str]]):
        """
        Create a Param_value instance from a dictionary.
        :param param_dict: Dictionary containing 'value' and 'unit'.
        :param value_key: Key in the dictionary to extract the value.
        :return: Param_value instance.
        """
        if value_key not in param_dict:
            raise KeyError(f"Key '{value_key}' not found in the provided dictionary.")

        if isinstance(param_dict[value_key][0], tuple):
            value = param_dict[value_key][0]
            unit = param_dict[value_key][1]
            return tuple(cls(v, unit) for v in value)

        elif isinstance(param_dict[value_key][0], (float, int)):
            value = float(np.asarray(param_dict[value_key][0]).flatten()[0])  # Why ?
            unit = param_dict[value_key][1]
            return cls(value, unit)
        else:
            raise TypeError(
                f"Value for '{value_key}' must be a int, float or tuple, got {type(param_dict[value_key][0])} instead."
            )

    @classmethod
    def from_quantity(cls, quantity):
        # Handles units conversion.
        obj = object.__new__(cls)
        obj.quantity = quantity
        return obj

    def _get_quantity(self, other):
        if isinstance(other, Param_value):
            return other.quantity
        return other * ureg.dimensionless

    # Arithmetic
    def __add__(self, other):
        return Param_value.from_quantity(self.quantity + self._get_quantity(other))

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        return Param_value.from_quantity(self.quantity - self._get_quantity(other))

    def __rsub__(self, other):
        return Param_value.from_quantity(self._get_quantity(other) - self.quantity)

    def __mul__(self, other):
        return Param_value.from_quantity(self.quantity * self._get_quantity(other))

    def __rmul__(self, other):
        return self.__mul__(other)

    def __truediv__(self, other):
        return Param_value.from_quantity(self.quantity / self._get_quantity(other))

    def __rtruediv__(self, other):
        return Param_value.from_quantity(self._get_quantity(other) / self.quantity)

    def __pow__(self, other):
        return Param_value.from_quantity(self.quantity**other)

    # NumPy support
    def __array__(self, dtype=None):
        """Convert to a NumPy array with the specified dtype."""
        return np.array(self.quantity, dtype=dtype)

    def __array_ufunc__(self, ufunc, method, *inputs, **kwargs):
        def unwrap(val):
            """Unwraps Param_value instances to their underlying quantity."""
            if isinstance(val, Param_value):
                # if instance is a Param_value object, return the pint quantity
                return val.quantity

            elif isinstance(val, (list, tuple, np.ndarray)):
                # if instance is a list, tuple or numpy array, unwrap each element
                return np.vectorize(unwrap)(val)
            else:
                return val

        unwrapped_inputs = [
            i.quantity if isinstance(i, Param_value) else i for i in inputs
        ]
        result = getattr(ufunc, method)(*unwrapped_inputs, **kwargs)

        def wrap(val):
            if isinstance(val, pint.Quantity):
                """Wraps a pint Quantity back into a Param_value."""
                return Param_value.from_quantity(val)
            return val

        if isinstance(result, pint.Quantity):
            return wrap(result)
        elif isinstance(result, (list, tuple, np.ndarray)) and isinstance(
            result.flat[0], pint.Quantity
        ):
            # If the result is a list, tuple or numpy array of pint Quantities, wrap each element
            return np.vectorize(wrap)(result)
        else:
            return result

    @classmethod
    def array_from_quantities(cls, quantities: np.ndarray, unit: str = None):
        """Convert a NumPy array of pint.Quantity to Param_value instances."""
        wrap = np.vectorize(lambda v: cls(v, unit), otypes=[object])
        return wrap(quantities)

    def round(self, ndigits=None):
        """Round the value to a specified number of decimal places."""
        if ndigits is None:
            return Param_value.from_quantity(self.quantity.round())
        else:
            return Param_value.from_quantity(self.quantity.round(ndigits))

    # ✅ Conversion to another compatible unit
    def to(self, new_unit: str, *args, **kwargs):
        """Convert to a different compatible unit (e.g., 'mol/m**3')."""
        try:
            R = kwargs.get(
                "R", 8.31446261815324 * ureg.joule / (ureg.mol * ureg.kelvin)
            )
            T = kwargs.get("T", 298.15 * ureg.kelvin)  # Default temperature in Kelvin

            if self.unit == "Pa" and new_unit in ["mol/m**3", "mol/m^3"]:
                # Convert pressure to molar concentration using ideal gas law
                converted = self.quantity / (R * T)

            elif self.unit in ["mol/m**3", "mol/m^3"] and new_unit == "Pa":
                # Convert molar concentration to pressure using ideal gas law
                converted = self.quantity * (R * T)

            else:
                # Default pint-based unit conversion
                converted = self.quantity.to(new_unit)

            return Param_value.from_quantity(converted).to_base_units()

        except pint.DimensionalityError as e:
            raise ValueError(f"Cannot convert from {self.unit} to {new_unit}: {e}")

    # ✅ Convert to base SI units (e.g. 'μmol/m^3' → 'mol/m^3')
    def to_base_units(self):
        return Param_value.from_quantity(self.quantity.to_base_units())

    # ✅ Force dimensionless conversion (e.g. For °C in Tetens formula)
    def force_dimensionless(self):
        """Convert to a dimensionless quantity."""
        self.quantity = self.quantity / 1 * ureg.degC

    # Comparison helpers
    def _compare(self, other, op):
        if isinstance(other, Param_value):
            try:
                other_converted = other.quantity.to(self.quantity.units)
            except pint.DimensionalityError:
                raise ValueError("Cannot compare quantities with incompatible units.")
            return op(self.quantity.magnitude, other_converted.magnitude)
        elif isinstance(other, (int, float, bool)):
            return op(self.quantity.magnitude, other)
        elif isinstance(other, ureg.Quantity):
            try:
                other_converted = other.to(self.quantity.units)
            except pint.DimensionalityError:
                raise ValueError("Cannot compare quantities with incompatible units.")
            return op(self.quantity.magnitude, other_converted.magnitude)
        else:
            return NotImplemented

    # Rich comparisons
    def __eq__(self, other):
        return self._compare(other, lambda a, b: a == b)

    def __ne__(self, other):
        return self._compare(other, lambda a, b: a != b)

    def __lt__(self, other):
        return self._compare(other, lambda a, b: a < b)

    def __le__(self, other):
        return self._compare(other, lambda a, b: a <= b)

    def __gt__(self, other):
        return self._compare(other, lambda a, b: a > b)

    def __ge__(self, other):
        return self._compare(other, lambda a, b: a >= b)

In [ ]:


def calculate_temperatures(
    F: float, E_mes: float, param_dict: Dict[str, float], solver_method: str = "hybr"
) -> Tuple[float, float, float]:
    """
    Solve for temperatures (T_mes, T_bs, T_soil) using a nonlinear solver.
    T_soil is a single soil compartment including surface, root zone, and former sink.
    """

    def get_scalar(value):
        return float(np.asarray(value).flatten()[0])

    # Extract scalar parameters from the dictionary and store them in Param_value instances
    T_air = Param_value.set_from_dict("T_air", param_dict)
    gamma_mes = Param_value.set_from_dict("gamma_mes", param_dict)
    gamma_bs = Param_value.set_from_dict("gamma_bs", param_dict)
    gamma_soil = Param_value.set_from_dict("gamma_soil", param_dict)
    h_mes = Param_value.set_from_dict("h_mes", param_dict)
    h_soil = Param_value.set_from_dict("h_soil", param_dict)
    k_soil = Param_value.set_from_dict("k_soil", param_dict)
    d_soil = Param_value.set_from_dict("d_soil", param_dict)
    C_w = Param_value.set_from_dict("C_w", param_dict)
    khi = Param_value.set_from_dict("khi", param_dict)

    alpha_mes = Param_value.set_from_dict("alpha_mes", param_dict)
    alpha_bs = Param_value.set_from_dict("alpha_bs", param_dict)
    alpha_surf = Param_value.set_from_dict("alpha_surf", param_dict)

    tau_mes = Param_value.set_from_dict("tau_mes", param_dict)
    tau_bs = Param_value.set_from_dict("tau_bs", param_dict)

    phi = Param_value.set_from_dict("phi", param_dict)
    zeta = Param_value.set_from_dict("zeta", param_dict)
    E_surf = Param_value.set_from_dict("E_surf", param_dict)

    A_mes = Param_value.set_from_dict("A_mes", param_dict)
    A_bs = Param_value.set_from_dict("A_bs", param_dict)
    A_root = Param_value.set_from_dict("A_root", param_dict)
    A_contact = Param_value.set_from_dict("A_contact", param_dict)

    U_mes_bs = Param_value.set_from_dict("U_mes_bs", param_dict)
    A_circ = Param_value(
        min(A_bs.value, A_root.value), param_dict["A_bs"][1]
    )  # Don't use the setter here

    phi_mes = alpha_mes * phi * zeta
    phi_bs = alpha_bs * tau_mes * phi * zeta
    phi_soil = alpha_surf * ((zeta * tau_bs * tau_mes) + (1 - zeta)) * phi

    # Effective soil conductance from thermal conductivity and depth
    U_soil = k_soil / d_soil
    beta_soil = h_soil + U_soil

    def residuals(T):
        T_mes, T_bs, T_soil = T
        z = zeta if zeta > 0 else 0.0

        eq1 = (
            A_mes * phi_mes
            - A_mes * gamma_mes * T_mes
            - A_mes * h_mes * (T_mes - T_air)
            - U_mes_bs * A_contact * (T_mes - T_bs)
            - khi * E_mes
        )

        eq2 = z * (
            A_bs * phi_bs
            - A_bs * gamma_bs * T_bs
            + U_mes_bs * A_contact * (T_mes - T_bs)
            - C_w * F * A_circ * (T_bs - T_soil)
        )

        eq3 = (
            phi_soil
            - gamma_soil * T_soil
            - beta_soil * (T_soil - T_air)
            - khi * E_surf
            + C_w * F * A_circ * (T_bs - T_soil)
        )

        return np.array([eq1, eq2, eq3], dtype=np.float64)

    initial_guess = np.random.uniform(
        low=[temp_lower_bound, temp_lower_bound, temp_lower_bound],
        high=[temp_upper_bound, temp_upper_bound, temp_upper_bound],
        size=(3,),
    )
    lower_bounds = np.array([250, 250, 250])
    upper_bounds = np.array([350, 350, 350])

    if solver_method == "least_squares":
        sol = least_squares(
            residuals,
            initial_guess,
            bounds=(lower_bounds, upper_bounds),
            method="trf",
            xtol=1e-6,
            ftol=1e-6,
            gtol=1e-6,
        )
        if not sol.success:
            raise ValueError(
                f"Temperature solver (least_squares) did not converge: {sol.message}"
            )
        T_mes, T_bs, T_soil = sol.x
    elif solver_method == "hybr":
        sol = root(residuals, initial_guess, method="hybr", tol=1e-6)
        if not sol.success:
            raise ValueError(
                f"Temperature solver (hybr) did not converge: {sol.message}"
            )
        T_mes, T_bs, T_soil = np.clip(sol.x, lower_bounds, upper_bounds)
    else:
        raise ValueError(
            "Invalid solver_method specified. Use 'least_squares' or 'hybr'."
        )

    if zeta == 0:
        T_mes = np.nan
        T_bs = np.nan

    return T_mes, T_bs, T_soil

In [104]:
#

# Equations for water

Considering the following flow of water in the system(mesophyll, bundle sheath and soil) ($mol/s/m^2$):

1. $$F_{soil-bs} = \dfrac{K_{soil}}{\mu} \cdot(\psi_{soil} - \psi_{air}) $$
2. $$F_{bs-mes} = \dfrac{K_{bs}}{\mu} \cdot(\psi_{bs} - \psi_{mes})$$
3. $$E_{mes} = \dfrac{K_{mes}}{\mu} \cdot(\psi_{mes} - \psi_{air})$$
4. $$F = c^{ste}$$

considering the following water quantities

1. $R$: rainfall (mol/s/$m^2$)
2. $L$ leakage
3. $S$: run off
   The water balance equations for th system form by mesophyll, bundle sheath and soil are:
4. Mes: $\dfrac{dw*{mes}}{dt} = F*{bs-mes} -E\_{mes} $
5. Bs: $ \dfrac{dw*{bs}}{dt}= F*{soil-bs} - F - F\_{bs-mes}$
6. Soil : $\dfrac{dw_{soil}}{dt}  = R - L - S- E_{soil} + F$

At the steady state the model become:

1. Mes: $F*{bs-mes} -E*{mes}=0 $
2. Bs: $F_{soil-bs} - F - F_{bs-mes=0}$
3. Soil : $\ R - L - S-  E_{soil}l + F=0$


In [105]:
# ChatGPT
# Parameters
# Vp_max = Param_value.set_from_dict("Vp_max", param_dict)   # mol / m⁻² s⁻¹
# Kp = Param_value.set_from_dict("Kp_25", param_dict)        # mol / m³
# Vc_max = Param_value.set_from_dict("Vcmax_25", param_dict)  # mol / m⁻² s⁻¹
# Kc = Param_value.set_from_dict("Kc_25", param_dict)        # mol / m³
# phi = 0.2     # leakiness
# gm = 2        # mesophyll conductance

# # Given ambient CO2 (Ca)
# Ca = np.linspace(200, 800, 100)
# Vp_values = []  # to store PEPC rates
# Vc_values = []  # to store Rubisco rates
# for Cai in Ca:

#     # Estimate mesophyll CO2 (Cm)
#     Cm = Cai - (Vp_max / gm)

#     # PEPC rate
#     Vp = Vp_max * Cm / (Kp + Cm)
#     Vp_values.append(Vp)
#     # Decarboxylation rate
#     Vdec = Vp  # assume all PEPC product is decarboxylated

#     # Rubisco rate (with leakiness)
#     Vc = (1 - phi) * Vdec
#     Cbs = Vc * Kc / (Vc_max - Vc)  # from inverted Michaelis-Menten
#     Vc_calc = Vc_max * Cbs / (Kc + Cbs)
#     Vc_values.append(Vc_calc)

# # Convert lists to numpy arrays for plotting
# Vp_values = np.array(Vp_values)
# Vc_values = np.array(Vc_values)
# #plotting
# plt.figure(figsize=(10, 6))
# plt.plot(Ca, Vp_values, label='PEPC Rate (Vp)', color='blue')
# plt.plot(Ca, Vc_values, label='Rubisco Rate (Vc)', color='green')
# plt.title('PEPC and Rubisco Rates vs Ambient CO2')
# plt.xlabel('Ambient CO2 (Ca) [μmol mol⁻¹]')
# plt.ylabel('Rate [μmol m⁻² s⁻¹]')
# plt.legend()
# plt.grid()

In [106]:
# vpmax_T = lambda Ea_Vpmax, T, R, DS_Vpmax, Hd_Vpmax, Vpmax_25 : \
#     Vpmax_25 * (np.exp(np.clip(Ea_Vpmax * (T - 298.15) / 298.15 * R * T, -500, 500))) \
#     * ((1+ np.exp((298.15 * DS_Vpmax - Hd_Vpmax)/(298.15 * R))) \
#        / (1 + np.exp((T * DS_Vpmax - Hd_Vpmax) / T*R)))

# Kp_T = lambda Kp_25, Ea_Kp, T, R : \
#     Kp_25 * np.exp(np.clip(Ea_Kp * (T - 298.15) / (298.15 * R * T), -500, 500))

# def calculate_pepc_rate(Ea_Vpmax, T, R, DS_Vpmax, Hd_Vpmax, Vpmax_25, Kp_25, Ea_Kp, PEPC, HCO3) :

#     return 1e6*(
#         (vpmax_T(Ea_Vpmax, T, R, DS_Vpmax, Hd_Vpmax, Vpmax_25) *
#         PEPC * HCO3) / np.clip(Kp_T(Kp_25, Ea_Kp, T, R) + HCO3, 1e-9, 1e6)
#     )

In [107]:
# CA_rate = lambda kcat_CA, CA, CO2_mes, K_CA : \
# (kcat_CA * CA * CO2_mes) / (K_CA + CO2_mes)

# A_rubisco_25 = lambda R, CO2_bs, O2_bs, Sco25, Kc_25, Ko_25, Vcmax_25, rubisco : \
#     (Vcmax_25 * rubisco * CO2_bs - (0.5 * O2_bs / Sco25) * 1e-3) \
#     / np.clip(CO2_bs + Kc_25 * (1 + O2_bs / Ko_25), 1e-6, 1e6)

# def calculate_and_plot_Vp_Vc(param_dict, CO2_values) :

#     R = 8.314  # J mol⁻¹ K⁻¹
#     T = 273.15+25  # Temperature in Kelvin
#     Ea_Vpmax = Param_value.set_from_dict("Ea_Vpmax", param_dict)  # Activation energy for Vpmax
#     DS_Vpmax = Param_value.set_from_dict("DS_Vpmax", param_dict)  # Entropy term for Vpmax
#     Hd_Vpmax = Param_value.set_from_dict("Hd_Vpmax", param_dict)  # Enthalpy term for Vpmax
#     Vpmax_25 = Param_value.set_from_dict("Vpmax_25", param_dict)  # Vpmax at 25°C
#     Kp_25 = Param_value.set_from_dict("Kp_25", param_dict)  # Kp at 25°C
#     Ea_Kp = Param_value.set_from_dict("Ea_K_HCO3", param_dict)  # Activation energy for Kp

#     kcat_CA = Param_value.set_from_dict("kcat_CA", param_dict)
#     K_CA = Param_value.set_from_dict("K_CA", param_dict)


#     Sco25 = Param_value.set_from_dict("Sco_25", param_dict)
#     Kc_25 = Param_value.set_from_dict("Kc_25", param_dict)
#     Ko_25 = Param_value.set_from_dict("Ko_25", param_dict)
#     Vcmax_25 = Param_value.set_from_dict("Vcmax_25", param_dict)

#     PEPC = Param_value.set_from_dict("PEPC", param_dict)  # PEPC concentration (arbitrary units)
#     CA = Param_value.set_from_dict("CA", param_dict)  # Carbonic anhydrase concentration (arbitrary units)
#     #HCO3 = 1.0  # Bicarbonate concentration (arbitrary units)

#     Vp_values = []
#     Vc_values = []

#     for CO2 in CO2_values:
#         CO2_mes = CO2 * 0.8  # Mesophyll CO2 concentration
#         HCO3 = CA_rate(kcat_CA, CA, CO2_mes, K_CA)  # Ambient CO2 concentration

#         Vp_rate = calculate_pepc_rate(
#             Ea_Vpmax, T, R, DS_Vpmax, Hd_Vpmax, Vpmax_25, Kp_25, Ea_Kp, PEPC, HCO3
#         )
#         Vc_rate = A_rubisco_25(
#             R, CO2_mes, CO2_mes, Sco25, Kc_25, Ko_25, Vcmax_25,
#             rubisco=1.0  # Rubisco concentration (arbitrary units)
#         )

#         Vp_values.append(Vp_rate)
#         Vc_values.append(Vc_rate)

#     # Plotting
#     plt.figure(figsize=(10, 6))
#     plt.plot(CO2_values, Vp_values, label='PEPC Rate (Vp)', color='blue')
#     plt.plot(CO2_values, Vc_values, label='Rubisco Rate (Vc)', color='green')
#     plt.title('PEPC and Rubisco Rates vs CO2 Concentration')
#     plt.xlabel('CO2 Concentration [μmol mol⁻¹]')
#     plt.ylabel('Rate [μmol m⁻² s⁻¹]')
#     plt.legend()
#     plt.grid()

# calculate_and_plot_Vp_Vc(param_dict, np.linspace(200, 800, 100))

In [108]:
# R = Param_value.set_from_dict("R", param_dict)
# PEPC = Param_value.set_from_dict("PEPC", param_dict)
# HCO3 = Param_value.set_from_dict("bicarbonate", param_dict)


# Vpmax_25 = Param_value.set_from_dict("Vpmax_25", param_dict)
# Ea_Vpmax = Param_value.set_from_dict("Ea_Vpmax", param_dict)
# DS_Vpmax = Param_value.set_from_dict("DS_Vpmax", param_dict)
# Hd_Vpmax = Param_value.set_from_dict("Hd_Vpmax", param_dict)

# Kp_25 = Param_value.set_from_dict("K_HCO3_25", param_dict)
# Ea_Kp = Param_value.set_from_dict("Ea_K_HCO3", param_dict)
# T_mes = Param_value(273.15 + 30, "degK")  # Example temperature in Kelvin
# Tk = T_mes
# T_base = Param_value(298.15, "degK")

# exp_Vpmax = np.exp(np.clip(Ea_Vpmax * (Tk - T_base) / (T_base * R * Tk), -500, 500))
# deactivation = (1 + np.exp((T_base * DS_Vpmax - Hd_Vpmax) / (T_base * R))) / (
#     1 + np.exp((Tk * DS_Vpmax - Hd_Vpmax) / (Tk * R))
# )
# Vpmax_T = Vpmax_25 * exp_Vpmax * deactivation
# print(exp_Vpmax)

In [109]:
def dpepc_rate(T_mes, param_dict):
    Vpmax_25 = Param_value.set_from_dict("Vpmax_25", param_dict)
    Ea_Vpmax = Param_value.set_from_dict("Ea_Vpmax", param_dict)
    DS_Vpmax = Param_value.set_from_dict("DS_Vpmax", param_dict)
    Hd_Vpmax = Param_value.set_from_dict("Hd_Vpmax", param_dict)
    Kp_25 = Param_value.set_from_dict("K_HCO3_25", param_dict)
    Ea_Kp = Param_value.set_from_dict("Ea_K_HCO3", param_dict)
    HCO3 = Param_value.set_from_dict("bicarbonate", param_dict)
    T = T_mes  # Temperature in Kelvin
    T_base = Param_value(273.15 + 25, "degK")  # Base temperature in Kelvin

    B = (
        PEPC
        * Vpmax_25
        * (np.exp((DS_Vpmax * T_base - Hd_Vpmax) / (R * T_base)) + 1)
        * np.exp(Ea_Vpmax * (T - T_base) / (R * T * T_base))
    )
    C = (
        0.001
        * Kp_25
        * np.exp(Ea_Kp * (T - T_base) / (R * T * T_base))
        * (np.exp((DS_Vpmax * T - Hd_Vpmax) / (R * T)) + 1)
    )
    X = HCO3

    return (B * C) / (C + X) ** 2

In [110]:
def pepc_assimilation_rate(T_mes: float, param_dict: Dict[str, float]) -> float:
    """
    PEPC assimilation rate following Boyd et al. (2015).
    """
    R = Param_value.set_from_dict("R", param_dict)
    PEPC = Param_value.set_from_dict("PEPC", param_dict)
    HCO3_Pa = Param_value.set_from_dict("bicarbonate", param_dict)
    HCO3 = HCO3_Pa.to("mol/m**3")  # Convert to mol/m³ if needed
    print(f"{HCO3}; {HCO3_Pa}")
    Vpmax_25 = Param_value.set_from_dict("Vpmax_25", param_dict)
    Ea_Vpmax = Param_value.set_from_dict("Ea_Vpmax", param_dict)
    DS_Vpmax = Param_value.set_from_dict("DS_Vpmax", param_dict)
    Hd_Vpmax = Param_value.set_from_dict("Hd_Vpmax", param_dict)

    Kp_25 = Param_value.set_from_dict("K_HCO3_25", param_dict)
    Ea_Kp = Param_value.set_from_dict("Ea_K_HCO3", param_dict)

    Tk = T_mes
    T_base = Param_value(298.15, "degK")  # Base temperature in Kelvin
    exp_Vpmax = np.exp(np.clip(Ea_Vpmax * (Tk - T_base) / (T_base * R * Tk), -500, 500))
    deactivation = (1 + np.exp((T_base * DS_Vpmax - Hd_Vpmax) / (T_base * R))) / (
        1 + np.exp((Tk * DS_Vpmax - Hd_Vpmax) / (Tk * R))
    )
    Vpmax_T = Vpmax_25 * exp_Vpmax * deactivation

    Kp_T = (
        Kp_25
        * np.exp(np.clip(Ea_Kp * (Tk - T_base) / (T_base * R * Tk), -500, 500))
        * 1e-3
    )

    denom = np.clip(Kp_T + HCO3, 1e-9, 1e6)

    rate = (Vpmax_T * PEPC * HCO3) / denom

    return 1e6 * rate


def rubisco_assimilation_rate(T_bs: float, param_dict: Dict[str, float]) -> float:
    """
    Rubisco assimilation rate with high-temperature deactivation following Boyd et al. (2015).
    """
    R = Param_value.set_from_dict("R", param_dict)
    rubisco = Param_value.set_from_dict("rubisco", param_dict)
    carbon = Param_value.set_from_dict("carbon", param_dict)
    oxygen = Param_value.set_from_dict("oxygen", param_dict)

    Vcmax_25 = Param_value.set_from_dict("Vcmax_25", param_dict)
    Ea_Vcmax = Param_value.set_from_dict("Ea_Vcmax", param_dict)
    DS_Vcmax = Param_value.set_from_dict(
        "DS_Vcmax", param_dict
    )  # Entropy term for Rubisco (new parameter)
    Hd_Vcmax = Param_value.set_from_dict(
        "Hd_Vcmax", param_dict
    )  # Heat of deactivation for Rubisco (new parameter)

    Kc_25 = Param_value.set_from_dict("Kc_25", param_dict)
    Ea_Kc = Param_value.set_from_dict("Ea_Kc", param_dict)
    Ko_25 = Param_value.set_from_dict("Ko_25", param_dict)
    Ea_Ko = Param_value.set_from_dict("Ea_Ko", param_dict)
    Sco_25 = Param_value.set_from_dict("Sco_25", param_dict)
    Ea_Sco = Param_value.set_from_dict("Ea_Sco", param_dict)

    Tk = T_bs

    # Vcmax with high-temperature deactivation
    exp_Vcmax = np.exp(np.clip(Ea_Vcmax * (Tk - 298.15) / (298.15 * R * Tk), -500, 500))
    deactivation = (1 + np.exp((298.15 * DS_Vcmax - Hd_Vcmax) / (298.15 * R))) / (
        1 + np.exp((Tk * DS_Vcmax - Hd_Vcmax) / (Tk * R))
    )
    Vcmax_T = Vcmax_25 * exp_Vcmax * deactivation

    # Michaelis constants temperature dependence
    Kc_T = (
        Kc_25
        * np.exp(np.clip(Ea_Kc * (Tk - 298.15) / (298.15 * R * Tk), -500, 500))
        * 1e-3
    )
    Ko_T = (
        Ko_25
        * np.exp(np.clip(Ea_Ko * (Tk - 298.15) / (298.15 * R * Tk), -500, 500))
        * 1e-3
    )
    Sco_T = Sco_25 * np.exp(
        np.clip(Ea_Sco * (Tk - 298.15) / (298.15 * R * Tk), -500, 500)
    )

    net_CO2 = carbon - (0.5 * oxygen / Sco_T) * 1e-3
    denom = np.clip(carbon + Kc_T * (1 + oxygen / Ko_T), 1e-6, 1e6)

    A_rubisco = (Vcmax_T * rubisco * net_CO2) / denom

    return 1e6 * A_rubisco if np.isfinite(A_rubisco) else np.nan

In [111]:
# constants = param_dict["constants"]
# tetens_a = Param_value.set_from_dict("tetens_a", constants)  #
# tetens_b = Param_value.set_from_dict("tetens_b", constants)  #
# tetens_c = Param_value.set_from_dict("tetens_c", constants)  #
# T_celsius = T_air.to("delta_degC")
# e_s = (tetens_a * np.exp((tetens_b * T_celsius)/ (T_celsius + tetens_c)))
# print(f"tetens_a: {tetens_a}, tetens_b: {tetens_b}, tetens_c: {tetens_c}")

In [112]:
# phi = Param_value.set_from_dict("phi", param_dict)
# a = Param_value.set_from_dict("a_value", param_dict)  # Slope for temperature regression
# b = Param_value.set_from_dict("b_value", param_dict)  # Intercept for temperature regression

# T_air = a * phi + b  # Air temperature [K]


# T_celsius = T_air.to("degC")
# print(T_celsius.round(2))

In [113]:
import numpy as np
from typing import Tuple, Dict


def objective(vars: Tuple[float, float], param_dict: Dict[str, float]) -> float:
    """
    Objective function for optimizing C4 photosynthesis temperature response.

    This model incorporates:
    - Temperature-dependent PEPC and Rubisco kinetics
    - Bi-substrate PEPC activation by bicarbonate
    - Enzyme deactivation at supra-optimal temperatures
    - Transpiration based on vapor pressure deficit (Pa-based formulation)

    Parameters
    ----------
    vars : Tuple[float, float]
        Tuple containing:
        - g_s (float): Mesophyll conductance [mol m^-2 s^-1 Pa^-1]
        - F (float): Water flux or other control variable

    param_dict : Dict[str, float]
        Dictionary of environmental, physiological, and physical constants.

    Returns
    -------
    float
        Negative Rubisco-limited assimilation rate [mol m^-2 s^-1] for minimization.
    """
    g_s = Param_value(
        vars[0], "mol m^-2 s^-1 Pa^-1"
    )  # Mesophyll conductance [mol m^-2 s^-1 Pa^-1]
    F = Param_value(
        vars[1], "kg m^-2 s^-1"
    )  # Water flux or other control variable [kg m^-2 s^-1]

    # try:
    # Unpack constants
    constants = param_dict["constants"]
    # === Gas diffusion parameters ===
    CO2_air = Param_value.set_from_dict("CO2_air", param_dict)  # External CO2 [Pa]
    O2_air = Param_value.set_from_dict("O2_air", param_dict)  # External O2 [Pa]
    g_bs = Param_value.set_from_dict(
        "g_bs", param_dict
    )  # Bundle sheath conductance [mol m^-2 s^-1 Pa^-1]
    g_m = Param_value.set_from_dict(
        "g_m", param_dict
    )  # Mesophyll conductance [mol m^-2 s^-1 Pa^-1]
    # === Environmental parameters ===
    phi = param_dict["phi"][0]  # dimensionality error with phi a and b WIP
    a = param_dict["a_value"][0]  # Slope for temperature regression
    b = param_dict["b_value"][0]  # Intercept for temperature regression
    # phi = Param_value.set_from_dict("phi", param_dict)  # Arbitrary environmental driver
    # a = Param_value.set_from_dict("a_value", param_dict)  # Slope for temperature regression
    # b = Param_value.set_from_dict("b_value", param_dict)  # Intercept for temperature regression
    T_air = Param_value(a * phi + b, "degK")  # Air temperature [K]

    RH = Param_value.set_from_dict("RH", param_dict)  # Relative humidity [fraction 0-1]
    if RH > 1.0:
        RH /= 100.0  # Convert from % if necessary

    # Pressure : only used when using conductances without Pa units.
    if "pressure" in param_dict:
        pressure = Param_value.set_from_dict("pressure", param_dict)
    else:
        pressure = Param_value.set_from_dict(
            "P_atm", constants
        )  # Atmospheric pressure [Pa]

    M_water = Param_value.set_from_dict(
        "M_water", constants
    )  # Molar mass of water [kg/mol]

    # Convert air temperature to Celsius for Tetens equation
    T_celsius = T_air.to("degC").round(2).value  # Air temperature [°C]

    # Compute saturation vapor pressure using Tetens equation [Pa]
    e_s = (
        constants["tetens_a"][0]
        * np.exp(
            (constants["tetens_b"][0] * T_celsius)
            / (T_celsius + constants["tetens_c"][0])
        )
        * constants["kPa_to_Pa"]  # Convert kPa to Pa
    ) * ureg.Pa

    # Vapor Pressure Deficit [Pa]
    VPD = e_s * (1 - RH)

    # Transpiration rate [kg m^-2 s^-1]
    # g_s: mol m^-2 s^-1 Pa^-1 (stomatal conductance)
    # g_m: mol m^-2 s^-1 Pa^-1 (mesophyll conductance)
    # VPD / pressure: unitless pressure gradient fraction
    # M_water: kg/mol converts molar flux to mass flux
    g_air_mes = 1 / (1 / g_s + 1 / g_m)
    g_air_bs = 1 / (1 / g_s + 1 / g_m + 1 / g_bs)

    E_mes = g_air_mes * VPD * M_water  # kg m^-2 s^-1

    # Update param_dict with computed air temperature
    param_dict = param_dict.copy()
    param_dict["T_air"] = (T_air.value, str(T_air.unit))

    # === Solve thermodynamic heat model ===
    T_mes, T_bs, T_soil = calculate_temperatures(F, E_mes, param_dict)

    # === Gas diffusion gradients ===

    CO2_mes_init = (
        Param_value.set_from_dict("CO2_air", param_dict) * 0.9
    )  # Initial mesophyll CO2 [Pa] equals to 90% of air CO2 concentration
    O2_bs_init = Param_value.set_from_dict(
        "O2_bs_init", param_dict
    )  # Initial bundle sheath O2 [Pa]

    # Mesophyll CO2 and bundle sheath O2 [mol m^-2 s^-1]
    CO2_mes = g_air_mes * (
        CO2_air - CO2_mes_init
    )  # WIP : Units are not ok maybe (or maybe because steady state, they are)
    O2_bs = g_air_bs * (O2_air - O2_bs_init)  # WIP

    # === Carbonic Anhydrase reaction (CO2 to HCO3-) ===
    kcat_CA = Param_value.set_from_dict("kcat_CA", param_dict)
    K_CA = Param_value.set_from_dict("K_CA", param_dict) / 1e6  # Convert to mol units
    CA = Param_value.set_from_dict("CA", param_dict)  # Enzyme concentration [mol m^-2]

    # Bicarbonate concentration [mol m^-2 s^-1]
    HCO3 = (kcat_CA * CA * CO2_mes) / (K_CA + CO2_mes)

    # === PEPC kinetics (temperature-adjusted) ===
    R = Param_value.set_from_dict("R", constants)
    Vpmax_25 = Param_value.set_from_dict("Vpmax_25", param_dict)
    Ea_Vpmax = Param_value.set_from_dict("Ea_Vpmax", param_dict)
    DS_Vpmax = Param_value.set_from_dict("DS_Vpmax", param_dict)
    Hd_Vpmax = Param_value.set_from_dict("Hd_Vpmax", param_dict)
    Kp_25 = Param_value.set_from_dict("K_HCO3_25", param_dict)
    Ea_Kp = Param_value.set_from_dict("Ea_K_HCO3", param_dict)

    # PEPC temperature response
    exp_Vpmax = np.exp(
        np.clip(Ea_Vpmax * (T_mes - 298.15) / (298.15 * R * T_mes), -500, 500)
    )
    deactivation = (1 + np.exp((298.15 * DS_Vpmax - Hd_Vpmax) / (298.15 * R))) / (
        1 + np.exp((T_mes * DS_Vpmax - Hd_Vpmax) / (T_mes * R))
    )
    Vpmax_T = Vpmax_25 * exp_Vpmax * deactivation

    # Temperature-dependent Kp [mol m^-2 s^-1]
    Kp_T = (
        Kp_25
        * np.exp(np.clip(Ea_Kp * (T_mes - 298.15) / (298.15 * R * T_mes), -500, 500))
        * 1e-6  # Convert to mol units
    )

    # PEPC assimilation rate [mol m^-2 s^-1]
    A_pepc = (Vpmax_T * HCO3) / (Kp_T + HCO3)

    # === Rubisco-limited assimilation ===
    CO2_bs = A_pepc  # Estimated bundle sheath CO2 [mol m^-2 s^-1]

    # print(f"\n===\nPUMP CO2_bs : {CO2_bs} mol m^-2 s^-1")

    Vcmax_25 = Param_value.set_from_dict("Vcmax_25", param_dict)
    Ea_Vcmax = Param_value.set_from_dict("Ea_Vcmax", param_dict)
    DS_Vcmax = Param_value.set_from_dict("DS_Vcmax", param_dict)
    Hd_Vcmax = Param_value.set_from_dict("Hd_Vcmax", param_dict)
    Kc_25 = Param_value.set_from_dict("Kc_25", param_dict)
    Ea_Kc = Param_value.set_from_dict("Ea_Kc", param_dict)
    Ko_25 = Param_value.set_from_dict("Ko_25", param_dict)
    Ea_Ko = Param_value.set_from_dict("Ea_Ko", param_dict)
    Sco_25 = Param_value.set_from_dict("Sco_25", param_dict)
    Ea_Sco = Param_value.set_from_dict("Ea_Sco", param_dict)
    # Rubisco temperature response
    Vcmax_T = (
        Vcmax_25
        * np.exp(np.clip(Ea_Vcmax * (T_bs - 298.15) / (298.15 * R * T_bs), -500, 500))
        * (
            (1 + np.exp((298.15 * DS_Vcmax - Hd_Vcmax) / (298.15 * R)))
            / (1 + np.exp((T_bs * DS_Vcmax - Hd_Vcmax) / (T_bs * R)))
        )
    )

    # Rubisco Michaelis-Menten constants (temperature-adjusted) [Pa]
    Kc_T = Kc_25 * np.exp(Ea_Kc * (T_bs - 298.15) / (298.15 * R * T_bs)) * 1e-9
    Ko_T = Ko_25 * np.exp(Ea_Ko * (T_bs - 298.15) / (298.15 * R * T_bs)) * 1e-9

    # Specificity factor (temperature-adjusted)
    Sco_T = Sco_25 * np.exp(Ea_Sco * (T_bs - 298.15) / (298.15 * R * T_bs))

    # Net CO2 concentration for Rubisco reaction [mol m^-2 s^-1]
    net_CO2 = np.clip(CO2_bs - (0.5 * O2_bs / Sco_T) * 1e-6, 1e-9, None)
    denom = np.clip(CO2_bs + Kc_T * (1 + O2_bs / Ko_T), 1e-6, 1e6)

    # Rubisco assimilation rate [mol m^-2 s^-1]
    A_rubisco = (Vcmax_T * net_CO2) / denom

    objective = A_rubisco  # Total assimilation rate

    # Return negative assimilation for minimization
    return -objective if np.isfinite(objective) else 1e6
    # except Exception as e:
    #     print("Objective error:", e)
    #     return 1e6

In [114]:
import numpy as np
from typing import Tuple, Dict


def objective_nopump(vars: Tuple[float, float], param_dict: Dict[str, float]) -> float:
    """
    Objective function for optimizing C4 photosynthesis temperature response.

    This model incorporates:
    - Temperature-dependent PEPC and Rubisco kinetics
    - Bi-substrate PEPC activation by bicarbonate
    - Enzyme deactivation at supra-optimal temperatures
    - Transpiration based on vapor pressure deficit (Pa-based formulation)

    Parameters
    ----------
    vars : Tuple[float, float]
        Tuple containing:
        - g_s (float): Mesophyll conductance [mol m^-2 s^-1 Pa^-1]
        - F (float): Water flux or other control variable

    param_dict : Dict[str, float]
        Dictionary of environmental, physiological, and physical constants.

    Returns
    -------
    float
        Negative Rubisco-limited assimilation rate [mol m^-2 s^-1] for minimization.
    """
    g_s, F = vars

    try:
        # Unpack constants
        constants = param_dict["constants"]
        # === Gas diffusion parameters ===
        CO2_air = Param_value.set_from_dict("CO2_air", param_dict)  # External CO2 [Pa]
        O2_air = Param_value.set_from_dict("O2_air", param_dict)  # External O2 [Pa]
        g_bs = Param_value.set_from_dict(
            "g_bs", param_dict
        )  # Bundle sheath conductance [mol m^-2 s^-1 Pa^-1]
        g_m = Param_value.set_from_dict(
            "g_m", param_dict
        )  # Mesophyll conductance [mol m^-2 s^-1 Pa^-1]
        # === Environmental parameters ===
        phi = param_dict["phi"][0]  # dimensionality error with phi, a, b WIP
        a = param_dict["a_value"][0]  # Slope for temperature regression
        b = param_dict["b_value"][0]  # Intercept for temperature regression
        # phi = Param_value.set_from_dict("phi", param_dict)  # Arbitrary environmental driver
        # a = Param_value.set_from_dict("a_value", param_dict)  # Slope for temperature regression
        # b = Param_value.set_from_dict("b_value", param_dict)  # Intercept for temperature regression
        T_air = Param_value(a * phi + b, "degK")  # Air temperature [K]
        RH = Param_value.set_from_dict(
            "RH", param_dict
        )  # Relative humidity [fraction 0-1]

        if RH > 1.0:
            RH /= 100.0  # Convert from % if necessary

        if "pressure" in param_dict:
            pressure = Param_value.set_from_dict("pressure", param_dict)
        else:
            pressure = Param_value.set_from_dict(
                "P_atm", constants
            )  # Atmospheric pressure [Pa]

        M_water = Param_value.set_from_dict(
            "M_water", constants
        )  # Molar mass of water [kg/mol]

        # Convert air temperature to Celsius for Tetens equation
        T_celsius = T_air.to("degC").round(2).value  # Air temperature [°C]

        # Compute saturation vapor pressure using Tetens equation [Pa]
        e_s = (
            constants["tetens_a"][0]
            * np.exp(
                (constants["tetens_b"][0] * T_celsius)
                / (T_celsius + constants["tetens_c"][0])
            )
            * constants["kPa_to_Pa"]  # Convert kPa to Pa
        ) * ureg.Pa

        # Vapor Pressure Deficit [Pa]
        VPD = e_s * (1 - RH)

        # Transpiration rate [kg m^-2 s^-1]
        # g_m: mol m^-2 s^-1 Pa^-1
        # g_s: mol m^-2 s^-1 Pa^-1 (stomatal conductance)
        # VPD / pressure: unitless pressure gradient fraction
        # M_water: kg/mol converts molar flux to mass flux
        g_air_mes = 1 / (1 / g_s + 1 / g_m)
        g_air_bs = 1 / (1 / g_s + 1 / g_m + 1 / g_bs)
        E_mes = g_air_mes * VPD * M_water

        # Update param_dict with computed air temperature
        param_dict = param_dict.copy()
        param_dict["T_air"] = (T_air.value, str(T_air.unit))

        # === Solve thermodynamic heat model ===
        T_mes, T_bs, T_soil = calculate_temperatures(F, E_mes, param_dict)

        # === Gas diffusion gradients ===

        CO2_mes_init = (
            Param_value.set_from_dict("CO2_air", param_dict) * 0.9
        )  # Initial mesophyll CO2 [Pa] equals to 90% of air CO2 concentration
        O2_bs_init = Param_value.set_from_dict(
            "O2_bs_init", param_dict
        )  # Initial bundle sheath O2 [Pa]
        CO2_bs_init = Param_value.set_from_dict(
            "CO2_bs_init", param_dict
        )  # Initial bundle sheath CO2 [Pa]
        CO2_mes = g_air_mes * (CO2_air - CO2_mes_init)
        # Total conductance to O2
        O2_bs = g_air_bs * (O2_air - O2_bs_init)

        # === Carbonic Anhydrase reaction (CO2 to HCO3-) ===

        # === PEPC kinetics (temperature-adjusted) ===
        R = Param_value.set_from_dict("R", constants)

        # === Rubisco-limited assimilation ===
        CO2_bs = g_air_bs * (
            CO2_air - CO2_bs_init
        )  # >>What if there is no carbon pump and the CO2 has to diffuse to the Bundle sheath ?<<
        # print(f"NOPUMP : CO2_bs: {CO2_bs}\nCO2_mes: {CO2_mes}")
        Vcmax_25 = Param_value.set_from_dict("Vcmax_25", param_dict)
        Ea_Vcmax = Param_value.set_from_dict("Ea_Vcmax", param_dict)
        DS_Vcmax = Param_value.set_from_dict("DS_Vcmax", param_dict)
        Hd_Vcmax = Param_value.set_from_dict("Hd_Vcmax", param_dict)
        Kc_25 = Param_value.set_from_dict("Kc_25", param_dict)
        Ea_Kc = Param_value.set_from_dict("Ea_Kc", param_dict)
        Ko_25 = Param_value.set_from_dict("Ko_25", param_dict)
        Ea_Ko = Param_value.set_from_dict("Ea_Ko", param_dict)
        Sco_25 = Param_value.set_from_dict("Sco_25", param_dict)
        Ea_Sco = Param_value.set_from_dict("Ea_Sco", param_dict)

        # Rubisco temperature response
        Vcmax_T = (
            Vcmax_25
            * np.exp(
                np.clip(Ea_Vcmax * (T_bs - 298.15) / (298.15 * R * T_bs), -500, 500)
            )
            * (
                (1 + np.exp((298.15 * DS_Vcmax - Hd_Vcmax) / (298.15 * R)))
                / (1 + np.exp((T_bs * DS_Vcmax - Hd_Vcmax) / (T_bs * R)))
            )
        )

        # Rubisco Michaelis-Menten constants (temperature-adjusted) [Pa]
        Kc_T = Kc_25 * np.exp(Ea_Kc * (T_bs - 298.15) / (298.15 * R * T_bs)) * 1e-9
        Ko_T = Ko_25 * np.exp(Ea_Ko * (T_bs - 298.15) / (298.15 * R * T_bs)) * 1e-9

        # Specificity factor (temperature-adjusted)
        Sco_T = Sco_25 * np.exp(Ea_Sco * (T_bs - 298.15) / (298.15 * R * T_bs))

        # Net CO2 concentration for Rubisco reaction [mol m^-2 s^-1]
        net_CO2 = np.clip(CO2_bs - (0.5 * O2_bs / Sco_T) * 1e-6, 1e-9, None)
        denom = np.clip(CO2_bs + Kc_T * (1 + O2_bs / Ko_T), 1e-6, 1e6)

        # Rubisco assimilation rate [mol m^-2 s^-1]
        A_rubisco = (Vcmax_T * net_CO2) / denom

        # Return negative assimilation for minimization
        return -A_rubisco if np.isfinite(A_rubisco) else 1e6

    except Exception as e:
        print("Objective error:", e)
        return 1e6

In [115]:
import numpy as np
from typing import Tuple, Dict


def objective_mes(vars: Tuple[float, float], param_dict: Dict[str, float]) -> float:
    """
    Objective function for optimizing C4 photosynthesis temperature response.

    This model incorporates:
    - Temperature-dependent PEPC and Rubisco kinetics
    - Bi-substrate PEPC activation by bicarbonate
    - Enzyme deactivation at supra-optimal temperatures
    - Transpiration based on vapor pressure deficit (Pa-based formulation)

    Parameters
    ----------
    vars : Tuple[float, float]
        Tuple containing:
        - g_s (float): Mesophyll conductance [mol m^-2 s^-1 Pa^-1]
        - F (float): Water flux or other control variable

    param_dict : Dict[str, float]
        Dictionary of environmental, physiological, and physical constants.

    Returns
    -------
    float
        Negative Rubisco-limited assimilation rate [mol m^-2 s^-1] for minimization.
    """
    g_s, F = vars

    try:
        # Unpack constants
        constants = param_dict["constants"]
        # === Gas diffusion parameters ===
        CO2_air = Param_value.set_from_dict("CO2_air", param_dict)  # External CO2 [Pa]
        O2_air = Param_value.set_from_dict("O2_air", param_dict)  # External O2 [Pa]
        g_bs = Param_value.set_from_dict(
            "g_bs", param_dict
        )  # Bundle sheath conductance [mol m^-2 s^-1 Pa^-1]
        g_m = Param_value.set_from_dict(
            "g_m", param_dict
        )  # Mesophyll conductance [mol m^-2 s^-1 Pa^-1]
        # === Environmental parameters ===
        phi = param_dict["phi"][0]  # dimensionality error with phi, a, b WIP
        a = param_dict["a_value"][0]  # Slope for temperature regression
        b = param_dict["b_value"][0]  # Intercept for temperature regression
        # phi = Param_value.set_from_dict("phi", param_dict)  # Arbitrary environmental driver
        # a = Param_value.set_from_dict("a_value", param_dict)  # Slope for temperature regression
        # b = Param_value.set_from_dict("b_value", param_dict)  # Intercept for temperature regression
        T_air = Param_value(a * phi + b, "degK")  # Air temperature [K]

        RH = Param_value.set_from_dict(
            "RH", param_dict
        )  # Relative humidity [fraction 0-1]
        if RH > 1.0:
            RH /= 100.0  # Convert from % if necessary

        if "pressure" in param_dict:
            pressure = Param_value.set_from_dict("pressure", param_dict)
        else:
            pressure = Param_value.set_from_dict(
                "P_atm", constants
            )  # Atmospheric pressure [Pa]

        M_water = Param_value.set_from_dict(
            "M_water", param_dict
        )  # Molar mass of water [kg/mol]

        # Convert air temperature to Celsius for Tetens equation
        T_celsius = (
            T_air.to("degC").round(2).value
        )  # Air temperature [°C] (take only to value so that it is unitless)

        # Compute saturation vapor pressure using Tetens equation [Pa]
        e_s = (
            constants["tetens_a"][0]
            * np.exp(
                (constants["tetens_b"][0] * T_celsius)
                / (T_celsius + constants["tetens_c"][0])
            )
            * constants["kPa_to_Pa"]  # Convert kPa to Pa
        ) * ureg.Pa  # Convert to Pa with unit

        # Vapor Pressure Deficit [Pa]
        VPD = e_s * (1 - RH)

        # Transpiration rate [kg m^-2 s^-1]
        # g_m: mol m^-2 s^-1 Pa^-1
        # g_s: mol m^-2 s^-1 Pa^-1 (stomatal conductance)
        # VPD / pressure: unitless pressure gradient fraction
        # M_water: kg/mol converts molar flux to mass flux
        g_air_mes = 1 / (1 / g_s + 1 / g_m)
        g_air_bs = 1 / (1 / g_s + 1 / g_m + 1 / g_bs)

        E_mes = g_air_mes * VPD * M_water

        # Update param_dict with computed air temperature
        param_dict = param_dict.copy()
        param_dict["T_air"] = (T_air.value, str(T_air.unit))

        # === Solve thermodynamic heat model ===
        T_mes, T_bs, T_soil = calculate_temperatures(F, E_mes, param_dict)

        # === Gas diffusion gradients ===
        CO2_mes_init = (
            Param_value.set_from_dict("CO2_air", param_dict) * 0.9
        )  # Initial mesophyll CO2 [Pa] equals to 90% of air CO2 concentration
        O2_bs_init = Param_value.set_from_dict(
            "O2_bs_init", param_dict
        )  # Initial bundle sheath O2 [Pa]

        # Mesophyll CO2 and bundle sheath O2 [mol m^-2 s^-1]
        CO2_mes = g_air_mes * (CO2_air - CO2_mes_init)
        O2_bs = g_air_bs * (O2_air - O2_bs_init)

        # === Carbonic Anhydrase reaction (CO2 to HCO3-) ===

        # === PEPC kinetics (temperature-adjusted) ===
        R = Param_value.set_from_dict("R", constants)

        # === Rubisco-limited assimilation ===
        # CO2_bs = CO2_mes * g_bs # >>What if there is no carbon pump and the CO2 has to diffuse to the Bundle sheath ?<<
        # print(f"NOPUMP : CO2_bs: {CO2_bs}\nCO2_mes: {CO2_mes}")
        Vcmax_25 = Param_value.set_from_dict("Vcmax_25", param_dict)
        Ea_Vcmax = Param_value.set_from_dict("Ea_Vcmax", param_dict)
        DS_Vcmax = Param_value.set_from_dict("DS_Vcmax", param_dict)
        Hd_Vcmax = Param_value.set_from_dict("Hd_Vcmax", param_dict)
        Kc_25 = Param_value.set_from_dict("Kc_25", param_dict)
        Ea_Kc = Param_value.set_from_dict("Ea_Kc", param_dict)
        Ko_25 = Param_value.set_from_dict("Ko_25", param_dict)
        Ea_Ko = Param_value.set_from_dict("Ea_Ko", param_dict)
        Sco_25 = Param_value.set_from_dict("Sco_25", param_dict)
        Ea_Sco = Param_value.set_from_dict("Ea_Sco", param_dict)

        # Rubisco temperature response
        Vcmax_T = (
            Vcmax_25
            * np.exp(
                np.clip(Ea_Vcmax * (T_mes - 298.15) / (298.15 * R * T_mes), -500, 500)
            )
            * (
                (1 + np.exp((298.15 * DS_Vcmax - Hd_Vcmax) / (298.15 * R)))
                / (1 + np.exp((T_mes * DS_Vcmax - Hd_Vcmax) / (T_mes * R)))
            )
        )

        # Rubisco Michaelis-Menten constants (temperature-adjusted) [Pa]
        Kc_T = Kc_25 * np.exp(Ea_Kc * (T_mes - 298.15) / (298.15 * R * T_mes)) * 1e-9
        Ko_T = Ko_25 * np.exp(Ea_Ko * (T_mes - 298.15) / (298.15 * R * T_mes)) * 1e-9

        # Specificity factor (temperature-adjusted)
        Sco_T = Sco_25 * np.exp(Ea_Sco * (T_mes - 298.15) / (298.15 * R * T_mes))

        # Net CO2 concentration for Rubisco reaction [mol m^-2 s^-1]
        net_CO2 = np.clip(CO2_mes - (0.5 * O2_bs / Sco_T) * 1e-6, 1e-9, None)
        denom = np.clip(CO2_mes + Kc_T * (1 + O2_bs / Ko_T), 1e-6, 1e6)

        # Rubisco assimilation rate [mol m^-2 s^-1]
        A_rubisco = (Vcmax_T * net_CO2) / denom

        # Return negative assimilation for minimization
        return -A_rubisco if np.isfinite(A_rubisco) else 1e6

    except Exception as e:
        print("Objective error:", e)
        return 1e6

In [116]:
# def objective(vars: Tuple[float, float], param_dict: Dict[str, float]) -> float:
#     """
#     Composite objective function for steady-state C4 photosynthesis modeling.
#     Combines Rubisco assimilation with water-use and thermal penalties.

#     Parameters:
#         vars (Tuple[float, float]): Tuple of (g_m, F), where
#             g_m: stomatal conductance (0–1, dimensionless)
#             F: water flux from root to bundle sheath [kg/s·m²]
#         param_dict (Dict[str, float]): Parameter dictionary

#     Returns:
#         float: Penalized objective to minimize (-A_rubisco + penalties)
#     """
#     g_m, F = vars
#     try:
#         constants = param_dict["constants"]

#         # === Compute T_air from phi ===
#         phi = param_dict["phi"]
#         a = param_dict["a_value"]
#         b = param_dict["b_value"]
#         T_air = a * phi + b
#         param_dict = param_dict.copy()
#         param_dict["T_air"] = T_air

#         # === Compute VPD and E_mes ===
#         RH = param_dict["RH"]
#         pressure = param_dict.get("pressure", constants["P_atm"])
#         M_water = constants["M_water"]
#         R = param_dict["R"]
#         T_celsius = T_air - constants["kelvin_to_celsius"]
#         e_s = constants["tetens_a"] * np.exp((constants["tetens_b"] * T_celsius) / (T_celsius + constants["tetens_c"])) * constants["kPa_to_Pa"]
#         VPD = e_s * (1 - RH)
#         VPD_mol_m3 = VPD / (R * T_air)
#         E_mes = g_m * VPD_mol_m3 * M_water

#         # === Rainfall-based transpiration constraint ===
#         A_leaf = param_dict["A_leaf"]
#         rainfall_rate = param_dict["rainfall_rate"]
#         VPD_threshold = param_dict["VPD_threshold"]
#         E_max = max(rainfall_rate / A_leaf, 1e-12)
#         E_vpd_max = max(g_m * (VPD_threshold / pressure) * M_water, 1e-12)

#         # === Solve temperatures ===
#         T_mes, T_bs, T_surf, T_root = calculate_temperatures(F, E_mes, param_dict)

#         # === CO2 and O2 Gradients ===
#         CO2_air = param_dict["CO2_air"]
#         O2_air = param_dict["O2_air"]
#         g_bs = param_dict["g_bs"]
#         CO2_mes_init = param_dict["CO2_mes_init"]
#         O2_bs_init = param_dict["O2_bs_init"]

#         CO2_mes = g_m * (CO2_air - CO2_mes_init)
#         O2_bs = g_m * g_bs * (O2_air - O2_bs_init)

#         # === Carbonic Anhydrase and HCO3- Formation ===
#         kcat_CA = param_dict["kcat_CA"]
#         K_CA = param_dict["K_CA"] / 1e6
#         CA = param_dict["CA"]
#         HCO3 = (kcat_CA * CA * CO2_mes) / (K_CA + CO2_mes)

#         # === Light-dependent Electron Transport (temperature-dependent) ===
#         alpha = param_dict["alpha_mes"]
#         zeta = param_dict["zeta"]
#         theta = param_dict["theta"]
#         J_max25 = param_dict["J_max25"]
#         Ea_J = param_dict["Ea_J"]
#         DS_J = param_dict["DS_J"]
#         Hd_J = param_dict["Hd_J"]

#         I_abs = alpha * phi * zeta
#         J_max_T = J_max25 * np.exp(Ea_J * (T_mes - 298.15) / (298.15 * R * T_mes)) * \
#             (1 + np.exp((298.15 * DS_J - Hd_J) / (298.15 * R))) / \
#             (1 + np.exp((T_mes * DS_J - Hd_J) / (T_mes * R)))

#         a_q = theta
#         b_q = -(J_max_T + I_abs)
#         c_q = J_max_T * I_abs
#         J = (-b_q - np.sqrt(b_q**2 - 4 * a_q * c_q)) / (2 * a_q)

#         Q = J / 2
#         Vpmax_25 = param_dict["Vpmax_25"]
#         Ea_Vpmax = param_dict["Ea_Vpmax"]
#         DS_Vpmax = param_dict["DS_Vpmax"]
#         Hd_Vpmax = param_dict["Hd_Vpmax"]
#         Kp_25 = param_dict["K_HCO3_25"] / 1e6
#         Ea_Kp = param_dict["Ea_K_HCO3"]

#         Vpmax_T = Vpmax_25 * np.exp(Ea_Vpmax * (T_mes - 298.15) / (298.15 * R * T_mes)) * \
#             (1 + np.exp((298.15 * DS_Vpmax - Hd_Vpmax) / (298.15 * R))) / \
#             (1 + np.exp((T_mes * DS_Vpmax - Hd_Vpmax) / (T_mes * R)))
#         Kp_T = Kp_25 * np.exp(Ea_Kp * (T_mes - 298.15) / (298.15 * R * T_mes))

#         A_pepc = min(Q, (Vpmax_T * HCO3) / (Kp_T + HCO3))

#         # === Rubisco ===
#         Vcmax_25 = param_dict["Vcmax_25"]
#         Ea_Vcmax = param_dict["Ea_Vcmax"]
#         DS_Vcmax = param_dict["DS_Vcmax"]
#         Hd_Vcmax = param_dict["Hd_Vcmax"]
#         Kc_25 = param_dict["Kc_25"] / 1e6
#         Ea_Kc = param_dict["Ea_Kc"]
#         Ko_25 = param_dict["Ko_25"] / 1e6
#         Ea_Ko = param_dict["Ea_Ko"]
#         Sco_25 = param_dict["Sco_25"]
#         Ea_Sco = param_dict["Ea_Sco"]

#         CO2_bs = A_pepc

#         Vcmax_T = Vcmax_25 * np.exp(Ea_Vcmax * (T_bs - 298.15) / (298.15 * R * T_bs)) * \
#             (1 + np.exp((298.15 * DS_Vcmax - Hd_Vcmax) / (298.15 * R))) / \
#             (1 + np.exp((T_bs * DS_Vcmax - Hd_Vcmax) / (T_bs * R)))

#         Kc_T = Kc_25 * np.exp(Ea_Kc * (T_bs - 298.15) / (298.15 * R * T_bs)) * 1e-3
#         Ko_T = Ko_25 * np.exp(Ea_Ko * (T_bs - 298.15) / (298.15 * R * T_bs)) * 1e-3
#         Sco_T = Sco_25 * np.exp(Ea_Sco * (T_bs - 298.15) / (298.15 * R * T_bs))

#         net_CO2 = CO2_bs - (0.5 * O2_bs / Sco_T) * 1e-6
#         denom = np.clip(CO2_bs + Kc_T * (1 + O2_bs / Ko_T), 1e-6, 1e6)
#         A_rubisco = (Vcmax_T * net_CO2) / denom

#         # === Penalty Terms ===
#         T_max = param_dict["T_max"]
#         lambda_water = param_dict["lambda_water"]
#         lambda_vpd = param_dict["lambda_vpd"]
#         lambda_heat = param_dict["lambda_heat"]

#         penalty_water = lambda_water * (E_mes / E_max) ** 2
#         penalty_vpd = lambda_vpd * (E_mes / E_vpd_max) ** 2
#         penalty_heat = lambda_heat * ((T_mes / T_max) ** 4 + (T_bs / T_max) ** 4)

#         return -A_rubisco
#         #+ penalty_water + penalty_vpd + penalty_heat

#     except Exception as e:
#         print("Objective error:", e)
#         return 1e6

In [117]:
def find_stable_initial_guess(
    param_dict: Dict[str, float], obj_choice: str
) -> Tuple[float, float]:
    """
    Uses dual annealing to find stable initial guesses for g_s and F.

    Args:
        param_dict (Dict[str, float]): Dictionary containing system parameters.

    Returns:
        Tuple[float, float]: Initial guesses for decision variables (g_s, F),
        optimized using dual annealing.
    """
    param_copy = param_dict.copy()

    # ✅ Update T_air from phi
    phi = param_dict["phi"][0]
    a = param_dict["a_value"][0]
    b = param_dict["b_value"][0]
    # phi = Param_value.set_from_dict("phi", param_copy).value # dimensionality error with phi, a and b WIP
    # a = Param_value.set_from_dict("a_value", param_copy)
    # b = Param_value.set_from_dict("b_value", param_copy)
    T_air = Param_value(a * phi + b, "degK")
    param_copy["T_air"] = (T_air.value, str(T_air.unit))

    updated_bounds = [
        param_dict["g_s_BOUND"][0],
        param_dict["F_BOUND"][0],
    ]  # directly access the bounds from param_dict without using the pint wrapper.

    max_attempts = 3
    result = None
    for attempt in range(1, max_attempts + 1):
        current_maxiter = 1000 * attempt
        current_seed = 42 + attempt
        if obj_choice == "nopump":
            result = dual_annealing(
                lambda vars: objective_nopump(vars, param_copy),
                bounds=updated_bounds,
                maxiter=current_maxiter,
                seed=current_seed,
            )
        elif obj_choice == "mes":
            result = dual_annealing(
                lambda vars: objective_mes(vars, param_copy),
                bounds=updated_bounds,
                maxiter=current_maxiter,
                seed=current_seed,
            )
        else:
            result = dual_annealing(
                lambda vars: objective(vars, param_copy),
                bounds=updated_bounds,
                maxiter=current_maxiter,
                seed=current_seed,
            )

        if result.success:
            break
        else:
            logging.warning(
                "Attempt %d: Dual annealing did not converge: %s", attempt, result
            )

    if not result.success:
        logging.error(
            "Global optimization failed after %d attempts. Proceeding with last result.",
            max_attempts,
        )

    logging.info(
        f"Initial Guess - g_s: {result.x[0]:.5f}, F: {result.x[1]:.5e}, "
        f"Objective: {result.fun:.5f}, Iterations: {result.nit}"
    )
    print(
        f"Initial Guess - g_s: {result.x[0]:.5f}, F: {result.x[1]:.5e},\n"
        f"Objective: {result.fun:.5f}\n"
        f"Iterations: {result.nit}"
    )

    return result.x[0], result.x[1]


def refine_solution_with_lbfgsb(
    param_dict: Dict[str, float], g_s_init: float, F_init: float, obj_choice: str
) -> Tuple[float, float, float, float, float, float]:
    """
    Refines the solution using L-BFGS-B for local optimization.

    Returns:
        Tuple: Optimized g_s, F, T_mes, T_bs, T_soil, E_mes
    """
    param_copy = param_dict.copy()
    g_m = Param_value.set_from_dict(
        "g_m", param_copy
    )  # Mesophyll conductance [mol m^-2 s^-1 Pa^-1]

    # Dynamic T_air update based on phi
    phi = param_dict["phi"][0]
    a = param_dict["a_value"][0]
    b = param_dict["b_value"][0]
    # phi = Param_value.set_from_dict("phi", param_copy).value  # Environmental driver dimensionality error with phi, a and b WIP
    # a = Param_value.set_from_dict("a_value", param_copy)  # Slope for temperature regression
    # b = Param_value.set_from_dict("b_value", param_copy)  # Intercept for temperature regression
    T_air = Param_value(a * phi + b, "degK")  # Air temperature [K]
    param_copy["T_air"] = (T_air.value, str(T_air.unit))  # Store T_air in param_copy

    RH = Param_value.set_from_dict("RH", param_copy)  # Relative humidity [fraction 0-1]
    if RH > 1.0:
        RH /= 100.0

    constants = Param_value.set_from_dict("constants", param_copy)
    pressure = param_copy.get(
        "pressure", constants["P_atm"]
    )  # Atmospheric pressure [Pa]
    M_water = Param_value.set_from_dict(
        "M_water", constants
    )  # Molar mass of water [kg/mol]
    T_celsius = T_air.to("degC").round(2).value  # Air temperature [°C but unitless]

    e_s = (
        constants["tetens_a"][0]
        * np.exp(
            (constants["tetens_b"][0] * T_celsius)
            / (T_celsius + constants["tetens_c"][0])
        )
        * constants["kPa_to_Pa"]
        * ureg.Pa
    )  # Saturation vapor pressure [Pa]
    VPD = e_s * (1 - RH)
    bounds = [param_dict["g_s_BOUND"][0], param_dict["F_BOUND"][0]]

    max_attempts = 3
    result = None
    for attempt in range(1, max_attempts + 1):
        if obj_choice == "nopump":

            result = scipy_minimize(
                lambda vars: objective(vars, param_copy),
                x0=[g_s_init, F_init],
                method="L-BFGS-B",
                bounds=bounds,
                options={
                    "ftol": 1e-8,
                    "gtol": 1e-8,
                    "maxiter": 1000 * attempt,
                    "eps": 1e-10,
                },
            )
        elif obj_choice == "mes":
            result = scipy_minimize(
                lambda vars: objective_mes(vars, param_copy),
                x0=[g_s_init, F_init],
                method="L-BFGS-B",
                bounds=bounds,
                options={
                    "ftol": 1e-8,
                    "gtol": 1e-8,
                    "maxiter": 1000 * attempt,
                    "eps": 1e-10,
                },
            )
        else:
            result = scipy_minimize(
                lambda vars: objective(vars, param_copy),
                x0=[g_s_init, F_init],
                method="L-BFGS-B",
                bounds=bounds,
                options={
                    "ftol": 1e-8,
                    "gtol": 1e-8,
                    "maxiter": 1000 * attempt,
                    "eps": 1e-10,
                },
            )
        if result.success:
            break
        else:
            logging.warning("Attempt %d: L-BFGS-B failed: %s", attempt, result.message)

    if not result.success:
        logging.error(
            "Local optimization failed after %d attempts. Proceeding anyway.",
            max_attempts,
        )

    g_s_opt, F_opt = result.x

    # Recompute E_mes
    T_celsius = (
        T_air.to("degC").round(2).value
    )  # Air temperature [°C] (take only to value so that it is unitless)
    constants["tetens_a"][0] * np.exp(
        (constants["tetens_b"][0] * T_celsius) / (T_celsius + constants["tetens_c"][0])
    ) * constants[
        "kPa_to_Pa"
    ] * ureg.Pa  # Pa
    VPD = e_s * (1 - RH)
    g_air_mes_opt = 1 / (1 / g_s_opt + 1 / g_m)
    E_mes = g_air_mes_opt * (VPD / pressure) * M_water

    # Solve for new temperatures using the updated 3-compartment model
    T_mes, T_bs, T_soil = calculate_temperatures(F_opt, E_mes, param_copy)

    logging.info(
        f"Refined Solution - g_s: {g_s_opt:.5f}, with g_m : {g_air_mes_opt:.5f}, F: {F_opt:.5e}, "
        f"E_mes: {E_mes:.5e}, T_mes: {T_mes:.2f}, T_bs: {T_bs:.2f}, "
        f"T_soil: {T_soil:.2f}, Iter: {result.nit}"
    )
    print(
        f"Refined Solution - g_s: {g_s_opt:.5f}, with g_m : {g_air_mes_opt:.5f}, F: {F_opt:.5e},\n"
        f"E_mes: {E_mes:.5e}, T_mes: {T_mes:.2f}, T_bs: {T_bs:.2f},\n"
        f"T_soil: {T_soil:.2f}, Iter: {result.nit}"
    )

    return g_s_opt, F_opt, T_mes, T_bs, T_soil, E_mes


def optimize_with_global_local(
    param_dict: Dict[str, float], obj_choice: str
) -> Tuple[float, float, float, float, float, float]:
    """
    Combines global and local optimization to maximize Rubisco assimilation.

    Returns:
        Tuple:
            g_s, F, T_mes, T_bs, T_soil, E_mes
    """
    import logging

    param_copy = param_dict.copy()

    # Global search for stable initial guess
    g_s_init, F_init = find_stable_initial_guess(param_copy, obj_choice)

    # Local refinement
    g_s, F, T_mes, T_bs, T_soil, E_mes = refine_solution_with_lbfgsb(
        param_copy, g_s_init, F_init, obj_choice
    )

    logging.info(
        f"Final Optimized Values - g_s: {g_s:.5f}, F: {F:.5e}, "
        f"E_mes: {E_mes:.5e}, T_mes: {T_mes:.2f}, T_bs: {T_bs:.2f}, "
        f"T_soil: {T_soil:.2f}"
    )

    return g_s, F, T_mes, T_bs, T_soil, E_mes

In [118]:
def optimize_for_one_parameter(
    param_name: str,
    param_values: np.ndarray,
    param_dict: Dict[str, float],
    n_jobs: int = -1,
) -> List[Tuple[float, float, float, float, float, float]]:
    """
    Optimize for a single parameter while keeping all others constant.
    Uses global-local optimization to find optimal stomatal aperture (g_s ∈ [0, 1])
    and water flux (F), while computing resulting thermal and transpiration state.

    Returns:
        List of (g_s, F, T_mes, T_bs, T_soil, E_mes) for each parameter value.
    """

    def update_param_dict(
        param_dict: Dict[str, float], param_value: float
    ) -> Dict[str, float]:
        param_dict_copy = param_dict.copy()
        param_unit = param_dict_copy[param_name][1]
        param_dict_copy[param_name] = (param_value, param_unit)

        # Also update T_air if phi is varied
        if param_name == "phi":
            T_air_unit = param_dict_copy["T_air"][1]
            param_value_nodim = param_value.value
            param_dict_copy["T_air"] = (
                param_dict_copy["a_value"][0] * param_value_nodim
                + param_dict["b_value"][0],
                T_air_unit,  # dimensionality error with a, b and phi WIP
            )

        return param_dict_copy

    def single_optimization(
        index: int, param_value: float, objective: str = "objective"
    ) -> Tuple[int, Tuple[float, float, float, float, float, float]]:
        """
        Perform optimization for a single parameter value.

        Returns:
            Tuple containing the index and a 6-element result:
            (g_s, F, T_mes, T_bs, T_soil, E_mes)
        """
        try:
            param_dict_copy = update_param_dict(param_dict, param_value)
            optimized_values = optimize_with_global_local(param_dict_copy, objective)

            g_s, F, T_mes, T_bs, T_soil, E_mes = optimized_values

            logging.info(
                f"Optimized {param_name}={param_value:.5f} -> g_s: {g_s:.5f}, "
                f"F: {F:.5e}, T_mes: {T_mes:.2f}, T_bs: {T_bs:.2f}, "
                f"T_soil: {T_soil:.2f}, E_mes: {E_mes:.2e}"
            )

            return index, (g_s, F, T_mes, T_bs, T_soil, E_mes)

        except Exception as e:
            logging.error(f"Optimization failed for {param_name}={param_value}: {e}")
            return index, (
                float("nan"),
                float("nan"),
                float("nan"),
                float("nan"),
                float("nan"),
                float("nan"),
            )

    # Parallel optimization
    results = Parallel(n_jobs=n_jobs)(
        delayed(single_optimization)(index, param_value)
        for index, param_value in enumerate(
            tqdm(param_values, desc=f"Optimizing {param_name}")
        )
    )

    # Return results in correct order
    results = sorted(results, key=lambda x: x[0])

    return [res[1] for res in results]

In [119]:
# def optimize_for_two_parameters(
#     param1_name: str,
#     param2_name: str,
#     param1_values: np.ndarray,
#     param2_values: np.ndarray,
#     param_dict: Dict[str, float],
#     n_jobs: int = -1,
# ) -> List[Tuple[float, float, float, float, float, float]]:
#     """
#     Optimize for two parameters simultaneously while keeping all others constant.
#     Decision variables are g_m (stomatal aperture) and F (water flux).

#     Returns:
#         List of (g_m, F, T_mes, T_bs, T_soil, E_mes) for each (param1, param2) combination.
#     """

#     def update_param_dict(
#         param_dict: Dict[str, float], param1_value: float, param2_value: float
#     ) -> Dict[str, float]:
#         param_dict_copy = param_dict.copy()
#         param_dict_copy[param1_name] = param1_value
#         param_dict_copy[param2_name] = param2_value

#         # Adjust air temperature if solar radiation changes
#         if param1_name == "phi" or param2_name == "phi":
#             param_dict_copy["T_air"] = (
#                 param_dict["a_value"] * param_dict_copy["phi"] + param_dict["b_value"]
#             )

#         return param_dict_copy

#     def single_optimization(
#         index: int, param1_value: float, param2_value: float
#     ) -> Tuple[int, Tuple[float, float, float, float, float, float]]:
#         """
#         Perform optimization for a combination of two parameter values.

#         Returns:
#             Index and 6-tuple of optimized values: (g_m, F, T_mes, T_bs, T_soil, E_mes)
#         """
#         try:
#             param_dict_copy = update_param_dict(param_dict, param1_value, param2_value)
#             g_m, F, T_mes, T_bs, T_soil, E_mes = optimize_with_global_local(param_dict_copy)

#             logging.info(
#                 f"Optimized {param1_name}={param1_value:.5f}, {param2_name}={param2_value:.5f} -> "
#                 f"g_m: {g_m:.5f}, F: {F:.5e}, T_mes: {T_mes:.2f}, T_bs: {T_bs:.2f}, "
#                 f"T_soil: {T_soil:.2f}, E_mes: {E_mes:.2e}"
#             )

#             return index, (g_m, F, T_mes, T_bs, T_soil, E_mes)

#         except Exception as e:
#             logging.error(
#                 f"Optimization failed for {param1_name}={param1_value}, {param2_name}={param2_value}: {e}"
#             )
#             return index, (
#                 float("nan"),
#                 float("nan"),
#                 float("nan"),
#                 float("nan"),
#                 float("nan"),
#                 float("nan"),
#             )

#     # Generate combinations of parameter values
#     param_combinations = list(product(param1_values, param2_values))

#     # Parallel execution
#     results = Parallel(n_jobs=n_jobs)(
#         delayed(single_optimization)(idx, val1, val2)
#         for idx, (val1, val2) in enumerate(
#             tqdm(param_combinations, desc=f"Optimizing {param1_name} and {param2_name}")
#         )
#     )

#     # Preserve order
#     results = sorted(results, key=lambda x: x[0])

#     return [res[1] for res in results]

In [120]:
def compute_cooling_quantities(
    g_s_values: np.ndarray,
    F_opt_values: np.ndarray,
    T_mes_values: np.ndarray,
    T_bs_values: np.ndarray,
    T_soil_values: np.ndarray,
    param_dict: Dict[str, float],
) -> Dict[str, np.ndarray]:
    """
    Computes cooling quantities for the updated 3-compartment model.

    Args:
        g_s_values: Array of stomatal aperture values (0–1).
        F_opt_values: Water flow from root to bundle sheath [kg/s·m²].
        T_mes_values: Mesophyll temperature [K].
        T_bs_values: Bundle sheath temperature [K].
        T_soil_values: Unified soil compartment temperature [K].
        param_dict: Parameter dictionary.

    Returns:
        Dictionary of computed heat loss components and total absorbed energy.
    """
    param_dict_copy = param_dict.copy()
    constants = param_dict_copy["constants"]

    # Environmental parameters
    phi = Param_value.set_from_dict("phi", param_dict_copy)  # Solar radiation [W m^-2]
    T_air = Param_value.set_from_dict("T_air", param_dict_copy)  # Air temperature [K]
    zeta = Param_value.set_from_dict(
        "zeta", param_dict_copy
    )  # Fraction of absorbed radiation
    RH = Param_value.set_from_dict(
        "RH", param_dict_copy
    )  # Relative humidity [fraction 0-1]
    if "pressure" in param_dict_copy:
        pressure = Param_value.set_from_dict(
            "pressure", param_dict_copy
        )  # Atmospheric pressure [Pa]
    else:
        pressure = Param_value.set_from_dict("P_atm", constants)
    M_water = Param_value.set_from_dict(
        "M_water", constants
    )  # Molar mass of water [kg/mol]

    # Conductance parameters
    g_m = Param_value.set_from_dict(
        "g_m", param_dict_copy
    )  # Molar conductance of mesophyll [mol m^-2 s^-1 Pa^-1]
    g_m_values = np.full_like(g_s_values, g_m, dtype=np.float64)
    g_air_mes_values = 1 / (
        1 / g_s_values + 1 / g_m_values
    )  # Effective mesophyll conductance [mol m^-2 s^-1 Pa^-1]
    # VPD and transpiration
    T_celsius = T_air.to("degC").round(2).value  # Air temperature in Celsius (unitless)
    if RH > 1.0:
        RH /= 100.0

    e_s = (
        constants["tetens_a"][0]
        * np.exp(
            (constants["tetens_b"][0] * T_celsius)
            / (T_celsius + constants["tetens_c"][0])
        )
        * constants["kPa_to_Pa"]
        * ureg.Pa
    )  # Saturation vapor pressure [Pa]
    VPD = e_s * (1 - RH)

    E_mes_values = g_air_mes_values * (VPD / pressure) * M_water

    # Parameters
    U_mes_bs = Param_value.set_from_dict(
        "U_mes_bs", param_dict_copy
    )  # Conductance between mesophyll and bundle sheath [W m^-2 K^-1]
    h_mes = Param_value.set_from_dict(
        "h_mes", param_dict_copy
    )  # Convective heat transfer coefficient for mesophyll [W m^-2 K^-1]
    h_soil = Param_value.set_from_dict(
        "h_soil", param_dict_copy
    )  # Convective heat transfer coefficient for soil [W m^-2 K^-1]
    k_soil = Param_value.set_from_dict(
        "k_soil", param_dict_copy
    )  # Thermal conductivity of soil [W m^-1 K^-1]
    d_soil = Param_value.set_from_dict(
        "d_soil", param_dict_copy
    )  # Thickness of soil layer [m]
    gamma_mes = Param_value.set_from_dict(
        "gamma_mes", param_dict_copy
    )  # Emissivity of mesophyll
    gamma_bs = Param_value.set_from_dict(
        "gamma_bs", param_dict_copy
    )  # Emissivity of bundle sheath
    gamma_soil = Param_value.set_from_dict(
        "gamma_soil", param_dict_copy
    )  # Emissivity of soil
    kappa = Param_value.set_from_dict(
        "khi", param_dict_copy
    )  # Transpiration coefficient [mol m^-2 s^-1 Pa^-1]
    E_surf = Param_value.set_from_dict(
        "E_surf", param_dict_copy
    )  # Evaporation from soil surface [mol m^-2 s^-1]
    Cw = Param_value.set_from_dict(
        "C_w", param_dict_copy
    )  # Water heat capacity [J m^-2 K^-1]

    alpha_mes = Param_value.set_from_dict(
        "alpha_mes", param_dict_copy
    )  # Absorption coefficient for mesophyll
    alpha_bs = Param_value.set_from_dict(
        "alpha_bs", param_dict_copy
    )  # Absorption coefficient for bundle sheath
    alpha_surf = Param_value.set_from_dict(
        "alpha_surf", param_dict_copy
    )  # Absorption coefficient for soil surface
    tau_mes = Param_value.set_from_dict(
        "tau_mes", param_dict_copy
    )  # Transmittance of mesophyll
    tau_bs = Param_value.set_from_dict(
        "tau_bs", param_dict_copy
    )  # Transmittance of bundle sheath

    A_mes = Param_value.set_from_dict(
        "A_mes", param_dict_copy
    )  # Area of mesophyll [m^2]
    A_bs = Param_value.set_from_dict(
        "A_bs", param_dict_copy
    )  # Area of bundle sheath [m^2]
    A_root = Param_value.set_from_dict("A_root", param_dict_copy)  # Area of root [m^2]
    A_contact = Param_value.set_from_dict(
        "A_contact", param_dict_copy
    )  # Contact area between mesophyll and bundle sheath [m^2]
    A_circ = min(A_bs, A_root)  # Circular area for heat exchange [m^2]

    # Energy inputs
    phi_mes = zeta * alpha_mes * phi
    phi_bs = zeta * alpha_bs * tau_mes * phi
    phi_soil = alpha_surf * ((zeta * tau_bs * tau_mes) + (1 - zeta)) * phi
    total_absorbed = phi_mes + phi_bs + phi_soil

    valid_zeta = zeta > 0
    z = zeta if valid_zeta else 0.0
    beta_soil = h_soil + k_soil / d_soil

    cooling_quantities = {
        "absorbed_irradiance_mes": phi_mes,
        "absorbed_irradiance_bs": phi_bs,
        "absorbed_irradiance_soil": phi_soil,
        "rad_mes": gamma_mes * T_mes_values * z * A_mes,
        "rad_bs": gamma_bs * T_bs_values * z * A_bs,
        "rad_soil": gamma_soil * T_soil_values,
        "conv_mes": h_mes * (T_mes_values - T_air) * z * A_mes,
        "conv_soil": h_soil * (T_soil_values - T_air),
        "cond_mesbs": U_mes_bs * (T_mes_values - T_bs_values) * z * A_contact,
        "circular_flow": Cw * F_opt_values * (T_bs_values - T_soil_values) * A_circ,
        "cond_soil_air": beta_soil * (T_soil_values - T_air),
        "transp_mes": kappa * E_mes_values,
        "evap_soil": kappa * E_surf,
        "total_absorbed_irradiance": total_absorbed,
    }

    return cooling_quantities

In [121]:
def compute_total_irradiance(
    param_dict: Dict[str, float],
    vary_param: str,
    vary_values: List[float],
) -> np.ndarray:
    """
    Computes total absorbed irradiance across mesophyll, bundle sheath, and soil compartments.

    Args:
        param_dict (Dict[str, float]): Dictionary containing model parameters.
        vary_param (str): Name of the parameter to vary (e.g., 'phi').
        vary_values (List[float]): Values for the parameter to vary.

    Returns:
        np.ndarray: Total absorbed irradiance values (W/m²) for each varied parameter value.
    """
    absorbed_irradiance_values = np.zeros(len(vary_values))
    vary_param_unit = param_dict_copy[vary_param][1]

    for i, value in enumerate(vary_values):
        param_dict_copy = param_dict.copy()

        param_dict_copy[vary_param] = (value, vary_param_unit)

        # If varying phi, update T_air dynamically
        if vary_param == "phi":
            value_nodim = value.value
            param_dict_copy["T_air"] = (
                param_dict_copy["a_value"] * value_nodim
                + param_dict_copy[
                    "b_value"
                ],  # dimensionality error with a, b and phi WIP
                param_dict_copy["T_air"][1],
            )

        # Extract relevant parameters
        phi = Param_value.set_from_dict("phi", param_dict_copy)
        zeta = Param_value.set_from_dict("zeta", param_dict_copy)

        alpha_mes = Param_value.set_from_dict("alpha_mes", param_dict_copy)
        alpha_bs = Param_value.set_from_dict("alpha_bs", param_dict_copy)
        alpha_surf = Param_value.set_from_dict("alpha_surf", param_dict_copy)
        tau_mes = Param_value.set_from_dict("tau_mes", param_dict_copy)
        tau_bs = Param_value.set_from_dict("tau_bs", param_dict_copy)

        # Layer-specific absorbed irradiance (per m² soil surface)
        phi_mes = zeta * alpha_mes * phi
        phi_bs = zeta * alpha_bs * tau_mes * phi
        phi_soil = alpha_surf * ((zeta * tau_bs * tau_mes) + (1 - zeta)) * phi

        total_absorbed_irradiance = phi_mes + phi_bs + phi_soil
        absorbed_irradiance_values[i] = total_absorbed_irradiance

    return absorbed_irradiance_values

In [122]:
def plot_all_graphs(
    parameter_name: str,
    parameter_values: List[float],
    g_s_values: List[float],
    T_mes_values: List[float],
    T_bs_values: List[float],
    E_mes_values: List[float],
    F_opt_values: List[float],
    T_soil_values: List[float],
    total_irradiance_values: List[float],
    param_dict: Dict[str, float],
    param_unit: str = r"(W \cdot m^{-2})",
    error_T_mes_opt: float = 0.05,
    error_T_bs_opt: float = 0.05,
    error_E_mes_opt: float = 0.05,
    nrows: int = None,
    ncols: int = None,
) -> None:
    """
    Dynamically plots physiological and thermal responses for the 3-compartment model.
    """

    os.makedirs("plots", exist_ok=True)
    os.makedirs("figures", exist_ok=True)

    plots: List[Tuple[str, List[float], str, str]] = [
        (r"$g_s$", g_s_values, r"$g_s$ (dimensionless)", "black"),
        (r"$T_{mes}$", T_mes_values, r"$T_{mes} \, (K)$", "blue"),
        (r"$T_{bs}$", T_bs_values, r"$T_{bs} \, (K)$", "green"),
        (r"$T_{soil}$", T_soil_values, r"$T_{soil} \, (K)$", "orange"),
        (r"$E_{mes}$", E_mes_values, r"$E_{mes} \, (kg \cdot s^{-1})$", "purple"),
        (r"$F$", F_opt_values, r"$F \, (kg \cdot s^{-1})$", "cyan"),
        (
            r"Total Irradiance",
            total_irradiance_values,
            r"$\Sigma \phi \, (W \cdot m^{-2})$",
            "magenta",
        ),
    ]

    num_plots = len(plots)
    if nrows is None or ncols is None:
        ncols = min(3, num_plots)
        nrows = math.ceil(num_plots / ncols)

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()

    # Optional reference bands
    T_mes_opt_data = param_dict.get("T_mes_opt", None)
    if T_mes_opt_data is not None:
        T_mes_opt = Param_value(
            T_mes_opt_data[0], T_mes_opt_data[1]
        )  # Convert to Param_value if needed
    T_bs_opt_data = param_dict.get("T_bs_opt", None)
    if T_bs_opt_data is not None:
        T_bs_opt = Param_value(
            T_bs_opt_data[0], T_bs_opt_data[1]
        )  # Convert to Param_value if needed
    E_mes_opt_data = param_dict.get("E_mes_opt", None)
    if E_mes_opt_data is not None:
        E_mes_opt = Param_value(E_mes_opt_data[0], E_mes_opt_data[1])

    T_mes_band = (
        (T_mes_opt * (1 - error_T_mes_opt), T_mes_opt * (1 + error_T_mes_opt))
        if T_mes_opt
        else None
    )
    T_bs_band = (
        (T_bs_opt * (1 - error_T_bs_opt), T_bs_opt * (1 + error_T_bs_opt))
        if T_bs_opt
        else None
    )
    E_mes_band = (
        (E_mes_opt * (1 - error_E_mes_opt), E_mes_opt * (1 + error_E_mes_opt))
        if E_mes_opt
        else None
    )

    for idx, (title, values, ylabel, color) in enumerate(plots):
        ax = axes[idx]
        ax.plot(
            parameter_values,
            values,
            label=title,
            linestyle="-",
            marker="o",
            markersize=4,
            linewidth=2,
            color=color,
            markerfacecolor="white",
            markeredgewidth=1.2,
        )

        if title == r"$T_{mes}$" and T_mes_opt:
            ax.axhline(T_mes_opt, color="red", linestyle="--", label=r"$T_{mes_{opt}}$")
            ax.fill_between(parameter_values, *T_mes_band, color="red", alpha=0.2)

        if title == r"$T_{bs}$" and T_bs_opt:
            ax.axhline(
                T_bs_opt, color="darkorange", linestyle="--", label=r"$T_{bs_{opt}}$"
            )
            ax.fill_between(parameter_values, *T_bs_band, color="darkorange", alpha=0.2)

        if title == r"$E_{mes}$" and E_mes_opt:
            ax.axhline(
                E_mes_opt, color="teal", linestyle="--", label=r"$E_{mes_{opt}}$"
            )
            ax.fill_between(parameter_values, *E_mes_band, color="teal", alpha=0.2)

        ax.set_xlabel(rf"${{{parameter_name}}} \, {param_unit}$", fontsize=14)
        ax.set_ylabel(ylabel, fontsize=14)
        ax.set_title(title, fontsize=14, fontweight="bold", pad=6)
        ax.legend(fontsize=10, loc="best", frameon=False)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.tick_params(axis="both", which="major", labelsize=12)

    for idx in range(num_plots, len(axes)):
        fig.delaxes(axes[idx])

    plt.tight_layout()
    plt.savefig(f"plots/{parameter_name}_all_graphs.png", dpi=300, bbox_inches="tight")
    plt.savefig(
        f"figures/{parameter_name}_all_graphs.png", dpi=300, bbox_inches="tight"
    )
    plt.show()

In [123]:
def plot_summary(
    parameter_name: str,
    parameter_values: List[float],
    A_mes_values: List[float],
    A_bs_values: List[float],
    T_mes_values: List[float],
    T_bs_values: List[float],
    E_mes_values: List[float],
    F_opt_values: List[float],
    T_soil_values: List[float],
    param_dict: Dict[str, float],
    param_unit: str = r"(W \cdot m^{-2})",
    error_T_mes_opt: float = 0.05,
    error_T_bs_opt: float = 0.05,
    nrows: int = None,
    ncols: int = None,
    objective: str = "objective",
) -> None:
    """
    Plots a summary of the optimization results for the 3-compartment model.
    WORK IN PROGRESS, NEEDS TO BE GENERALIZED
    """

    plots: Dict[List[Tuple[str, List[float], str, str]]] = {
        "Mesophyll and Bundle sheath assimilation rate": [
            (r"$A_{mes}$", A_mes_values, r"$A_{mes} \, (mol.m^{-2}.s^{-1})$", "blue"),
            (r"$A_{bs}$", A_bs_values, r"$A_{bs} \, (mol.m^{-2}.s^{-1})$", "green"),
        ],
        "Mesophyll and Bundle sheath temperature": [
            (r"$T_{mes}$", T_mes_values, r"$T_{mes} \, (K)$", "darkviolet"),
            (r"$T_{bs}$", T_bs_values, r"$T_{bs} \, (K)$", "darkorange"),
            (r"$T_{soil}$", T_soil_values, r"$T_{soil} \, (K)$", "brown"),
        ],
        "Circular flow rate and evaporation rate": [
            (r"$E_{mes}$", E_mes_values, r"$E_{mes} \, (kg \cdot s^{-1})$", "purple"),
            (r"$F$", F_opt_values, r"$F \, (kg \cdot s^{-1})$", "cyan"),
        ],
    }

    figs, axes = plt.subplots(nrows=3, ncols=1, figsize=(5 * ncols, 10 * nrows))
    # Optional reference bands
    # T_mes_opt = param_dict.get("T_mes_opt", None)
    # T_bs_opt = param_dict.get("T_bs_opt", None)

    # T_mes_band = (
    #     (T_mes_opt * (1 - error_T_mes_opt), T_mes_opt * (1 + error_T_mes_opt))
    #     if T_mes_opt else None
    # )
    # T_bs_band = (
    #     (T_bs_opt * (1 - error_T_bs_opt), T_bs_opt * (1 + error_T_bs_opt))
    #     if T_bs_opt else None
    # )

    for idx, key in enumerate(plots.keys()):
        title = key
        values = plots[key]
        if "assimilation" in title.lower():
            ylabel = r"$Assimilation Rates \, (mol \cdot m^{-2} \cdot s^{-1})$"
        elif "temperature" in title.lower():
            ylabel = r"$Temperatures \, (K)$"
        elif "evaporation" in title.lower() or "circular flow" in title.lower():
            ylabel = r"$Transpiration and Water Flux \, (kg \cdot s^{-1})$"
        else:
            continue

        ax = axes[idx]
        for _, values, label, color in values:

            ax.plot(
                parameter_values,
                values,
                label=label,
                linestyle="-",
                marker="o",
                markersize=4,
                linewidth=2,
                color=color,
                markerfacecolor="white",
                markeredgewidth=1.2,
            )

        # if "temperature" in title.lower() and T_mes_opt and T_bs_opt:
        #     ax.axhline(T_mes_opt, color="violet", linestyle="--", label=None)
        #     ax.axhline(T_bs_opt, color="bisque", linestyle="--", label= None)
        # if T_mes_band:
        #     ax.fill_between(parameter_values, *T_mes_band, color="violet", alpha=0.2)
        # if T_bs_band:
        #     ax.fill_between(parameter_values, *T_bs_band, color="bisque", alpha=0.2)

        ax.set_xlabel(rf"${{{parameter_name}}} \, {param_unit}$", fontsize=14)
        ax.set_ylabel(ylabel, fontsize=14)
        ax.set_title(title, fontsize=14, fontweight="bold", pad=6)
        ax.legend(fontsize=10, loc="best", frameon=False)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.tick_params(axis="both", which="major", labelsize=12)
    if objective == "mes":
        plt.suptitle(
            f"Optimization Summary with rubisco in mesophyll",
            fontsize=16,
            fontweight="bold",
        )
    elif objective == "nopump":
        plt.suptitle(
            f"Optimization Summary C4 with KO PEPC", fontsize=16, fontweight="bold"
        )
    elif objective == "pump":
        plt.suptitle(
            f"Optimization Summary wild type C4", fontsize=16, fontweight="bold"
        )

    plt.tight_layout()
    plt.savefig(
        f"plots/{objective}_{parameter_name}_summary.png", dpi=300, bbox_inches="tight"
    )

In [124]:
def plot_all_graphs_A(
    parameter_name: str,
    parameter_values: List[float],
    g_s_values: List[float],
    A_mes_values: List[float],
    A_bs_values: List[float],
    T_mes_values: List[float],
    T_bs_values: List[float],
    E_mes_values: List[float],
    F_opt_values: List[float],
    T_soil_values: List[float],
    total_irradiance_values: List[float],
    T_air_values: List[float],
    param_dict: Dict[str, float],
    param_unit: str = r"(W \cdot m^{-2})",
    error_T_mes_opt: float = 0.05,
    error_T_bs_opt: float = 0.05,
    error_E_mes_opt: float = 0.05,
    nrows: int = None,
    ncols: int = None,
) -> None:
    """
    Dynamically plots physiological and thermal responses for the 3-compartment model.
    """

    os.makedirs("plots", exist_ok=True)
    os.makedirs("figures", exist_ok=True)

    plots: List[Tuple[str, List[float], str, str]] = [
        (r"$g_s$", g_s_values, r"$g_s$ (dimensionless)", "black"),
        (r"$A_{mes}$", A_mes_values, r"$A_{mes} \, (mol.m^{-2}.s^{-1})$", "blue"),
        (r"$A_{bs}$", A_bs_values, r"$A_{bs} \, (mol.m^{-2}.s^{-1})$", "green"),
        (r"$T_{mes}$", T_mes_values, r"$T_{mes} \, (K)$", "darkviolet"),
        (r"$T_{bs}$", T_bs_values, r"$T_{bs} \, (K)$", "darkorange"),
        (r"$T_{soil}$", T_soil_values, r"$T_{soil} \, (K)$", "orange"),
        (r"$E_{mes}$", E_mes_values, r"$E_{mes} \, (kg \cdot s^{-1})$", "purple"),
        (r"$F$", F_opt_values, r"$F \, (kg \cdot s^{-1})$", "cyan"),
        (r"$T_{air}$", T_air_values, r"$T_{air}, (K)$", "magenta"),
    ]

    num_plots = len(plots)
    if nrows is None or ncols is None:
        ncols = min(3, num_plots)
        nrows = math.ceil(num_plots / ncols)

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()

    # Optional reference bands
    T_mes_opt_data = param_dict.get("T_mes_opt", None)
    if T_mes_opt_data is not None:
        T_mes_opt = Param_value(
            T_mes_opt_data[0], T_mes_opt_data[1]
        )  # Convert to Param_value if needed
    T_bs_opt_data = param_dict.get("T_bs_opt", None)
    if T_bs_opt_data is not None:
        T_bs_opt = Param_value(
            T_bs_opt_data[0], T_bs_opt_data[1]
        )  # Convert to Param_value if needed
    E_mes_opt_data = param_dict.get("E_mes_opt", None)
    if E_mes_opt_data is not None:
        E_mes_opt = Param_value(E_mes_opt_data[0], E_mes_opt_data[1])

    T_mes_band = (
        (T_mes_opt * (1 - error_T_mes_opt), T_mes_opt * (1 + error_T_mes_opt))
        if T_mes_opt
        else None
    )
    T_bs_band = (
        (T_bs_opt * (1 - error_T_bs_opt), T_bs_opt * (1 + error_T_bs_opt))
        if T_bs_opt
        else None
    )
    E_mes_band = (
        (E_mes_opt * (1 - error_E_mes_opt), E_mes_opt * (1 + error_E_mes_opt))
        if E_mes_opt
        else None
    )

    # Mes_Bs_axes_index = []
    for idx, (title, values, ylabel, color) in enumerate(plots):
        #     if title == r"$A_{mes}$" and len(Mes_Bs_axes_index) > 0:
        #         if plots[Mes_Bs_axes_index[0]][0] == r"$T_{mes}$":
        #             print(f"Printing A_mes with T_mes on axis {Mes_Bs_axes_index[0]}")
        #             ax = axes[Mes_Bs_axes_index[0]].twinx()
        #         elif plots[Mes_Bs_axes_index[0]][1] == r"$T_{mes}$":
        #             ax = axes[Mes_Bs_axes_index[1]].twinx()
        #         else:
        #             ax = axes[idx]
        #             Mes_Bs_axes_index.append(idx)
        #     elif title == r"$A_{bs}$" and len(Mes_Bs_axes_index) > 0:
        #         if plots[Mes_Bs_axes_index[0]][0] == r"$T_{bs}$":
        #             ax = axes[Mes_Bs_axes_index[0]].twinx()
        #         elif plots[Mes_Bs_axes_index[0]][1] == r"$T_{bs}$":
        #             ax = axes[Mes_Bs_axes_index[1]].twinx()
        #         else:
        #             ax = axes[idx]
        #             Mes_Bs_axes_index.append(idx)
        #     else :
        #         ax = axes[idx]
        #         Mes_Bs_axes_index.append(idx)
        # print(f"====\n\nidx : {idx}; title : {title};\n values : {values}; \nylabel : {ylabel}; color : {color}\n\n====")
        ax = axes[idx]
        ax.plot(
            parameter_values,
            values,
            label=title,
            linestyle="-",
            marker="o",
            markersize=4,
            linewidth=2,
            color=color,
            markerfacecolor="white",
            markeredgewidth=1.2,
        )
        # ---^^^
        # if title == r"$T_{mes}$" and T_mes_opt:
        #     ax.axhline(T_mes_opt, color="red", linestyle="--", label=r"$T_{mes_{opt}}$")
        #     ax.fill_between(parameter_values, *T_mes_band, color="red", alpha=0.2)

        # if title == r"$T_{bs}$" and T_bs_opt:
        #     ax.axhline(T_bs_opt, color="darkorange", linestyle="--", label=r"$T_{bs_{opt}}$")
        #     ax.fill_between(parameter_values, *T_bs_band, color="darkorange", alpha=0.2)

        # if title == r"$E_{mes}$" and E_mes_opt:
        #     ax.axhline(E_mes_opt, color="teal", linestyle="--", label=r"$E_{mes_{opt}}$")
        #     ax.fill_between(parameter_values, *E_mes_band, color="teal", alpha=0.2)
        # ---vvv
        ax.set_xlabel(rf"${{{parameter_name}}} \, {param_unit}$", fontsize=14)
        ax.set_ylabel(ylabel, fontsize=14)
        ax.set_title(title, fontsize=14, fontweight="bold", pad=6)
        ax.legend(fontsize=10, loc="best", frameon=False)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.tick_params(axis="both", which="major", labelsize=12)
        if title == r"$A_{mes}$":
            plt.savefig(
                f"plots/{parameter_name}_A_mes.png", dpi=300, bbox_inches="tight"
            )
        if title == r"$A_{bs}$":
            plt.savefig(
                f"plots/{parameter_name}_A_bs.png", dpi=300, bbox_inches="tight"
            )

    for idx in range(num_plots, len(axes)):
        fig.delaxes(axes[idx])

    plt.tight_layout()
    plt.savefig(f"plots/{parameter_name}_all_graphs.png", dpi=300, bbox_inches="tight")
    plt.savefig(
        f"figures/{parameter_name}_all_graphs.png", dpi=300, bbox_inches="tight"
    )
    plt.show()

In [125]:
def generate_all_contour_plots(
    param1_name: str,
    param2_name: str,
    param1_values: np.ndarray,
    param2_values: np.ndarray,
    T_mes_values: np.ndarray,
    T_bs_values: np.ndarray,
    T_soil_values: np.ndarray,
    E_mes_values: np.ndarray,
    F_values: np.ndarray,
    total_irradiance_values: np.ndarray,
    cooling_quantities: Dict[str, np.ndarray],
    param_dict: Dict[str, float],
    param1_unit: str = r"(W \cdot m^{-2})",
    param2_unit: str = r"(W \cdot m^{-2})",
    nrows: int = None,
    ncols: int = None,
) -> None:
    """
    Generates contour plots for core thermal/physiological quantities and
    cooling mechanisms using the updated 3-compartment soil model.
    """

    os.makedirs("plots", exist_ok=True)

    core_plots: List[Tuple[np.ndarray, str, str]] = [
        (T_mes_values, r"$T_{mes}$", r"$T_{mes} \, (K)$"),
        (T_bs_values, r"$T_{bs}$", r"$T_{bs} \, (K)$"),
        (T_soil_values, r"$T_{soil}$", r"$T_{soil} \, (K)$"),
        (E_mes_values, r"$E_{mes}$", r"$E_{mes} \, (kg \cdot m^{-2} \cdot s^{-1})$"),
        (F_values, r"$F$", r"$F \,  (kg \cdot m^{-2} \cdot s^{-1})$"),
        (
            total_irradiance_values,
            r"Total Irradiance",
            r"$I_{abs} \, (W \cdot m^{-2})$",
        ),
    ]

    cooling_keys = [
        "rad_mes",
        "rad_bs",
        "rad_soil",
        "conv_mes",
        "conv_soil",
        "cond_mesbs",
        "circular_flow",
        "cond_soil_air",
        "transp_mes",
        "evap_soil",
    ]

    cooling_plots: List[Tuple[np.ndarray, str, str]] = [
        (
            cooling_quantities[key],
            key,
            key.replace("_", " ").title().replace("Soil", "Soil").replace("Bs", "BS"),
        )
        for key in cooling_keys
        if key in cooling_quantities
    ]

    def create_contour_subplot(plots_subset, filename_suffix):
        num_plots = len(plots_subset)
        ncols_local = min(3, num_plots) if ncols is None else ncols
        nrows_local = math.ceil(num_plots / ncols_local) if nrows is None else nrows

        fig, axes = plt.subplots(
            nrows=nrows_local,
            ncols=ncols_local,
            figsize=(5 * ncols_local, 4 * nrows_local),
        )
        axes = np.array(axes).flatten()
        X, Y = np.meshgrid(param1_values, param2_values)

        for idx, (Z, title, clabel) in enumerate(plots_subset):
            ax = axes[idx]
            contour = ax.contourf(X, Y, Z, levels=100, cmap="viridis")
            cbar = plt.colorbar(contour, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label(clabel, fontsize=12)
            ax.set_xlabel(rf"${{{param1_name}}} \, {param1_unit}$", fontsize=14)
            ax.set_ylabel(rf"${{{param2_name}}} \, {param2_unit}$", fontsize=14)
            ax.set_title(title, fontsize=14, fontweight="bold", pad=6)
            ax.grid(True, linestyle="--", alpha=0.6)
            ax.tick_params(axis="both", which="major", labelsize=12)

        for idx in range(num_plots, len(axes)):
            fig.delaxes(axes[idx])

        plt.tight_layout()
        plt.savefig(
            f"plots/{param1_name}_{param2_name}_{filename_suffix}.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()

    # Generate both sets of plots
    create_contour_subplot(core_plots, "core_variables")
    create_contour_subplot(cooling_plots, "cooling_quantities")

In [126]:
def plot_cooling_quantities_combined(
    parameter_name: str,
    parameter_values: List[float],
    cooling_quantities: Dict[str, np.ndarray],
    param_unit: str,
    file_name_prefix: str = "cooling_quantities",
    nrows: int = None,
    ncols: int = None,
) -> None:
    """
    Creates grouped subplots for cooling quantities categorized into absorbed irradiance,
    conductive cooling, convective cooling, radiative cooling, and transpiration & water flow.
    Includes a separate subplot for total absorbed irradiance if available.
    """

    os.makedirs("plots", exist_ok=True)
    os.makedirs("figures", exist_ok=True)

    def validate_and_reshape(values: List[float], length: int) -> np.ndarray:
        values = np.array(values)
        if values.ndim == 0 or values.size == 1:
            return np.full(length, values.item())
        if values.shape[0] != length:
            raise ValueError(f"Shape mismatch: expected {length}, got {values.shape}")
        return values.flatten()

    # Extract and temporarily remove total absorbed irradiance
    total_absorbed_irradiance = cooling_quantities.pop(
        "total_absorbed_irradiance", None
    )

    color_palette = plt.get_cmap("tab10")

    # Grouping categories
    grouped_quantities = {
        "Absorbed Irradiance": [],
        "Conductive Cooling": [],
        "Convective Cooling": [],
        "Radiative Cooling": [],
        "Transpiration & Flow": [],
    }

    for key in cooling_quantities.keys():
        if "absorbed_irr" in key:
            grouped_quantities["Absorbed Irradiance"].append(key)
        elif "cond" in key:
            grouped_quantities["Conductive Cooling"].append(key)
        elif "conv" in key:
            grouped_quantities["Convective Cooling"].append(key)
        elif "rad" in key:
            grouped_quantities["Radiative Cooling"].append(key)
        elif "transp" in key or "circ" in key:
            grouped_quantities["Transpiration & Flow"].append(key)

    # Remove empty categories
    grouped_quantities = {k: v for k, v in grouped_quantities.items() if v}

    total_groups = len(grouped_quantities)
    ncols = 3
    reserve_total_idx = 5
    extra_plot = 1 if total_absorbed_irradiance is not None else 0
    nrows = max(2, int(np.ceil((total_groups + extra_plot + 1) / ncols)))

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()

    plot_idx = 0
    for group_name, keys in grouped_quantities.items():
        if plot_idx == reserve_total_idx:
            plot_idx += 1  # Reserve spot for total irradiance

        ax = axes[plot_idx]
        num_keys = len(keys)
        colors = [color_palette(i % 10) for i in range(num_keys)]

        for i, key in enumerate(keys):
            y_values = validate_and_reshape(
                cooling_quantities[key], len(parameter_values)
            )
            label = key.replace("_", " ").title()
            ax.plot(
                parameter_values,
                y_values,
                label=label,
                linestyle="-",
                marker="o",
                markersize=4,
                linewidth=2,
                color=colors[i],
                markerfacecolor="white",
                markeredgewidth=1.2,
            )

        ax.set_title(group_name, fontsize=14, pad=6)
        ax.set_xlabel(rf"${parameter_name} \, {param_unit}$", fontsize=14)
        ax.set_ylabel(r"Cooling $\left(W \cdot m^{-2}\right)$", fontsize=14)
        ax.legend(fontsize=10, loc="best", frameon=False)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
        ax.tick_params(axis="both", which="major", labelsize=12)

        plot_idx += 1

    # Add total absorbed irradiance in reserved subplot
    if total_absorbed_irradiance is not None and len(axes) > reserve_total_idx:
        y_values = validate_and_reshape(
            total_absorbed_irradiance, len(parameter_values)
        )
        ax = axes[reserve_total_idx]
        ax.plot(
            parameter_values,
            y_values,
            label="Total Absorbed Irradiance",
            linestyle="-",
            marker="o",
            markersize=4,
            linewidth=2,
            color="black",
            markerfacecolor="white",
            markeredgewidth=1.2,
        )
        ax.set_title("Total Absorbed Irradiance", fontsize=14, pad=6)
        ax.set_xlabel(rf"${parameter_name} \, {param_unit}$", fontsize=14)
        ax.set_ylabel(r"$W \cdot m^{-2}$", fontsize=14)
        ax.legend(fontsize=10, loc="best", frameon=False)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
        ax.tick_params(axis="both", which="major", labelsize=12)

        # Restore the dictionary entry
        cooling_quantities["total_absorbed_irradiance"] = total_absorbed_irradiance

    # Remove unused axes
    for idx in range(plot_idx + 1, len(axes)):
        if idx != reserve_total_idx:
            fig.delaxes(axes[idx])

    plt.tight_layout()
    plt.subplots_adjust(top=0.9, hspace=0.5, wspace=0.4)
    output_path = f"plots/{file_name_prefix}.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.savefig(f"figures/{file_name_prefix}.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(f"✅ Plot successfully saved to: {output_path}")

In [127]:
def generate_contour_plot_cooling_quantities_combined(
    param1_name: str,
    param2_name: str,
    param1_values: np.ndarray,
    param2_values: np.ndarray,
    cooling_quantities: Dict[str, np.ndarray],
    param_unit1: str = r"(W \cdot m^{-2})",
    param_unit2: str = r"(unitless)",
    file_name_prefix: str = "cooling_quantities_combined",
) -> None:
    """
    Creates contour plots for each cooling quantity using the updated 3-compartment model.
    Saves all plots into one figure with a uniform layout and styling.
    """

    os.makedirs("plots", exist_ok=True)
    os.makedirs("figures", exist_ok=True)

    quantity_keys = list(cooling_quantities.keys())
    num_plots = len(quantity_keys)
    ncols = 3
    nrows = math.ceil(num_plots / ncols)

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()
    X, Y = np.meshgrid(param1_values, param2_values)

    for idx, key in enumerate(quantity_keys):
        Z = cooling_quantities[key]
        ax = axes[idx]

        contour = ax.contourf(X, Y, Z, levels=100, cmap="viridis")
        cbar = plt.colorbar(contour, ax=ax, fraction=0.046, pad=0.04)
        label_clean = (
            key.replace("_", " ").replace("bs", "BS").replace("soil", "Soil").title()
        )
        cbar.set_label(f"{label_clean} (W·m⁻²)", fontsize=12)

        ax.set_title(label_clean, fontsize=12, pad=6)
        ax.set_xlabel(rf"${{{param1_name}}} \, {param_unit1}$", fontsize=12)
        ax.set_ylabel(rf"${{{param2_name}}} \, {param_unit2}$", fontsize=12)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.tick_params(axis="both", which="major", labelsize=11)

    # Remove any extra axes
    for idx in range(num_plots, len(axes)):
        fig.delaxes(axes[idx])

    plt.tight_layout()
    plt.subplots_adjust(top=0.9, hspace=0.5, wspace=0.4)
    output_path = f"plots/{file_name_prefix}.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.savefig(f"figures/{file_name_prefix}.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(f"✅ Cooling quantity contour plots saved to: {output_path}")

In [128]:
def generate_all_contour_plots(
    param1_name: str,
    param2_name: str,
    param1_values: np.ndarray,
    param2_values: np.ndarray,
    T_mes_values: np.ndarray,
    T_bs_values: np.ndarray,
    T_soil_values: np.ndarray,
    E_mes_values: np.ndarray,
    F_values: np.ndarray,
    g_s_values: np.ndarray,
    cooling_quantities: Dict[str, np.ndarray],
    param_dict: Dict[str, float],
    param1_unit: str = r"(W \cdot m^{-2})",
    param2_unit: str = r"(W \cdot m^{-2})",
    nrows: int = None,
    ncols: int = None,
) -> None:
    """
    Generates contour plots for thermal, physiological, and cooling quantities using the updated soil model.
    """

    os.makedirs("plots", exist_ok=True)

    # === First Grid: Core physiological and thermal outputs ===
    core_plots: List[Tuple[np.ndarray, str, str]] = [
        (T_mes_values, r"$T_{mes}$", r"$T_{mes}$ (K)"),
        (T_bs_values, r"$T_{bs}$", r"$T_{bs}$ (K)"),
        (T_soil_values, r"$T_{soil}$", r"$T_{soil}$ (K)"),
        (E_mes_values, r"$E_{mes}$", r"$E_{mes}$  $(kg \cdot m^{-2} \cdot s^{-1})$"),
        (F_values, r"$F$", r"$F$ $(kg \cdot m^{-2} \cdot s^{-1})$"),
        (g_s_values, r"$g_s$", r"$g_s$ (unitless)"),
    ]

    def plot_grid(plots: List[Tuple[np.ndarray, str, str]], filename: str) -> None:
        num_plots = len(plots)
        local_ncols = min(3, num_plots) if ncols is None else ncols
        local_nrows = math.ceil(num_plots / local_ncols) if nrows is None else nrows

        fig, axes = plt.subplots(
            nrows=local_nrows,
            ncols=local_ncols,
            figsize=(5 * local_ncols, 4 * local_nrows),
        )
        axes = np.array(axes).flatten()
        X, Y = np.meshgrid(param1_values, param2_values)

        for idx, (Z, title, clabel) in enumerate(plots):
            ax = axes[idx]
            contour = ax.contourf(X, Y, Z, levels=100, cmap="viridis")
            cbar = plt.colorbar(contour, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label(clabel, fontsize=12)
            ax.set_xlabel(rf"${{{param1_name}}} \, {param1_unit}$", fontsize=12)
            ax.set_ylabel(rf"${{{param2_name}}} \, {param2_unit}$", fontsize=12)
            ax.set_title(title, fontsize=12, pad=6)
            ax.grid(True, linestyle="--", alpha=0.6)
            ax.tick_params(axis="both", which="major", labelsize=10)

        for idx in range(num_plots, len(axes)):
            fig.delaxes(axes[idx])

        plt.tight_layout()
        plt.savefig(f"plots/{filename}.png", dpi=300, bbox_inches="tight")
        plt.show()

    # === Plot core physiological outputs ===
    plot_grid(core_plots, f"{param1_name}_{param2_name}_core")

    # === Cooling quantities for 3-compartment model ===
    cooling_keys = [
        "absorbed_irradiance_mes",
        "absorbed_irradiance_bs",
        "absorbed_irradiance_soil",
        "rad_mes",
        "rad_bs",
        "rad_soil",
        "conv_mes",
        "conv_soil",
        "cond_mesbs",
        "circular_flow",
        "cond_soil_air",
        "transp_mes",
        "evap_soil",
    ]

    cooling_plots = [
        (
            cooling_quantities[k],
            k.replace("_", " ").title(),
            f"{k.replace('_', ' ').title()} (W·m⁻²)",
        )
        for k in cooling_keys
        if k in cooling_quantities
    ]

    if cooling_plots:
        plot_grid(cooling_plots, f"{param1_name}_{param2_name}_cooling")

In [129]:
def plot_total_irradiance(
    vary_values: np.ndarray,
    total_irradiance: np.ndarray,
    vary_param: str,
    param_unit: str = r"(W \cdot m^{-2})",
    file_name_prefix: str = "total_irradiance_",
) -> None:
    """
    Plots total absorbed irradiance versus a varied parameter and saves the figure.

    Args:
        vary_values (np.ndarray): Values of the varied parameter.
        total_irradiance (np.ndarray): Total absorbed irradiance values [W/m²].
        vary_param (str): Name of the varied parameter (e.g., 'phi', 'zeta').
        param_unit (str): Unit string for x-axis label.
        file_name_prefix (str): Prefix for saved figure filename.
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    os.makedirs("plots", exist_ok=True)

    if not isinstance(vary_values, np.ndarray) or not isinstance(
        total_irradiance, np.ndarray
    ):
        raise TypeError("Both vary_values and total_irradiance must be NumPy arrays.")

    if vary_values.shape != total_irradiance.shape:
        raise ValueError(
            f"Shape mismatch: {vary_values.shape} vs {total_irradiance.shape}"
        )

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.plot(
        vary_values,
        total_irradiance,
        label="Total Absorbed Irradiance",
        color="mediumseagreen",
        linestyle="-",
        linewidth=2.5,
        marker="o",
        markersize=7,
        markerfacecolor="white",
        markeredgewidth=1.5,
    )

    ax.set_xlabel(rf"${vary_param} \, {param_unit}$", fontsize=16, fontweight="bold")
    ax.set_ylabel(
        r"Total Absorbed Irradiance $(W \cdot m^{-2})$", fontsize=16, fontweight="bold"
    )

    ax.tick_params(axis="both", which="major", labelsize=14, width=1.5, length=6)
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)
    ax.legend(fontsize=14, frameon=False)

    plt.tight_layout()
    output_path = f"plots/{file_name_prefix}{vary_param}.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"✅ Plot successfully saved to: {output_path}")

In [130]:
def run_simulation_one_parameter(
    param_name: str,
    param_values: np.ndarray,
    param_dict: Dict[str, float],
    error_T_mes_opt: float = 0.05,
    error_T_bs_opt: float = 0.05,
    elev: float = 30,
    azim: float = 60,
    param_unit: str = r"(W \cdot m^{-2})",
    file_prefix: str = "one_param",
    a: float = 0.0,
    b: float = 300,
    nrows: int = 2,
    ncols: int = 3,
) -> Dict[str, List[float]]:
    """
    Runs a simulation across a range of one parameter and generates relevant plots
    for the updated 3-compartment energy model.
    """

    def update_param_dict(
        param_name: str, value: float, param_dict: Dict[str, float]
    ) -> Dict[str, float]:
        param_dict_copy = param_dict.copy()
        if param_name == "phi":
            value_nodim = value.value
            T_air_unit = param_dict_copy["T_air"].unit
            param_dict_copy["T_air"] = (
                a * value_nodim + b,
                T_air_value,
            )  # dimensionality error with a, b and phi WIP
        param_dict_copy[param_name] = value
        return param_dict_copy

    results = []
    T_air_values = [] if param_name == "phi" else None
    cooling_quantities_list = []

    for value in param_values:
        param_dict_copy = update_param_dict(param_name, value, param_dict)

        if param_name == "phi":
            T_air_values.append(param_dict_copy["T_air"])

        # === Run main model ===
        g_s, F, T_mes, T_bs, T_soil, E_mes = optimize_with_global_local(param_dict_copy)
        results.append((g_s, F, T_mes, T_bs, T_soil, E_mes))

        # === Cooling quantities ===
        cooling_quantities = compute_cooling_quantities(
            g_s_values=np.array([g_s]),
            F_opt_values=np.array([F]),
            T_mes_values=np.array([T_mes]),
            T_bs_values=np.array([T_bs]),
            T_soil_values=np.array([T_soil]),
            param_dict=param_dict_copy,
        )
        cooling_quantities_list.append(cooling_quantities)

    # ✅ Combine cooling quantities into arrays
    combined_cooling_quantities = {
        key: np.array(
            [
                cq[key] if np.isscalar(cq[key]) else cq[key][0]
                for cq in cooling_quantities_list
            ]
        )
        for key in cooling_quantities_list[0].keys()
    }

    # ✅ Unpack simulation outputs
    g_s_values, F_values, T_mes_values, T_bs_values, T_soil_values, E_mes_values = zip(
        *results
    )

    # ✅ Compute total irradiance
    total_irradiance = compute_total_irradiance(
        param_dict=param_dict,
        vary_param=param_name,
        vary_values=param_values,
    )

    # ✅ Plot model results
    plot_all_graphs_A(
        parameter_name=param_name,
        parameter_values=param_values,
        g_s_values=list(g_s_values),
        A_mes_values=[
            pepc_assimilation_rate(T_mes_values[i], param_dict)
            for i in range(len(param_values))
        ],
        A_bs_values=[
            rubisco_assimilation_rate(T_bs_values[i], param_dict)
            for i in range(len(param_values))
        ],
        T_mes_values=T_mes_values,
        T_bs_values=T_bs_values,
        E_mes_values=E_mes_values,
        F_opt_values=F_values,
        T_soil_values=T_soil_values,
        total_irradiance_values=total_irradiance,
        param_dict=param_dict,
        param_unit=param_unit,
        error_T_mes_opt=error_T_mes_opt,
        error_T_bs_opt=error_T_bs_opt,
    )

    # ✅ Plot cooling components
    plot_cooling_quantities_combined(
        parameter_name=param_name,
        parameter_values=param_values,
        cooling_quantities=combined_cooling_quantities,
        param_unit=param_unit,
        file_name_prefix=f"{file_prefix}_cooling_quantities",
        nrows=nrows,
        ncols=ncols,
    )

    return {
        "T_air": T_air_values if T_air_values is not None else None,
        "T_mes": list(T_mes_values),
        "T_bs": list(T_bs_values),
        "T_soil": list(T_soil_values),
        "E_mes": list(E_mes_values),
        "g_s": list(g_s_values),
        "F": list(F_values),
        "total_irradiance": total_irradiance.tolist(),
    }

In [131]:
def run_simulation_one_parameter_two_conditions(
    param_name: str,
    param_values: np.ndarray,
    param_dim: str,
    param_dict: Dict[str, float],
    error_T_mes_opt: float = 0.05,
    error_T_bs_opt: float = 0.05,
    elev: float = 30,
    azim: float = 60,
    param_unit: str = r"(W \cdot m^{-2})",
    file_prefix: str = "one_param",
    a: float = 0.0,
    b: float = 300,
    nrows: int = 2,
    ncols: int = 3,
) -> Dict[str, List[float]]:
    """
    Runs a simulation across a range of one parameter and generates relevant plots
    for the updated 3-compartment energy model.
    """

    def update_param_dict(
        param_name: str, value: float, unit: str, param_dict: Dict[str, float]
    ) -> Dict[str, float]:
        param_dict_copy = param_dict.copy()
        if param_name == "phi":
            T_air_unit = param_dict_copy["T_air"][1]
            param_dict_copy["T_air"] = (
                a * value + b,
                T_air_unit,
            )  # Dimensionality error to fix with a, b and phi WIP
        param_dict_copy[param_name] = (
            (value, unit)
            if isinstance(value, float) and isinstance(unit, str)
            else (value, None)
        )
        return param_dict_copy

    for obj_choice in ["pump", "nopump", "mes"]:
        results = []
        T_air_values = []
        cooling_quantities_list = []

        for value in param_values:
            # === Update parameter dictionary ===
            # This function updates the parameter dictionary with the current value of the parameter being varied.
            # If the parameter is "phi", it also updates the "T_air" value based on the linear relationship defined by 'a' and 'b'.
            # If the parameter is not "phi", Tair is not updated.
            param_dict_copy = update_param_dict(
                param_name, value, param_dim, param_dict
            )

            if param_name == "phi":

                T_air_values.append(Param_value.set_from_dict("T_air", param_dict_copy))
            else:
                T_air_values.append(Param_value.set_from_dict("T_air", param_dict_copy))

            # === Run main model ===
            g_s, F, T_mes, T_bs, T_soil, E_mes = optimize_with_global_local(
                param_dict_copy, obj_choice
            )
            results.append((g_s, F, T_mes, T_bs, T_soil, E_mes))

            # === Cooling quantities ===
            cooling_quantities = compute_cooling_quantities(
                g_s_values=np.array([g_s]),
                F_opt_values=np.array([F]),
                T_mes_values=np.array([T_mes]),
                T_bs_values=np.array([T_bs]),
                T_soil_values=np.array([T_soil]),
                param_dict=param_dict_copy,
            )
            cooling_quantities_list.append(cooling_quantities)

        # ✅ Combine cooling quantities into arrays
        combined_cooling_quantities = {
            key: np.array(
                [
                    cq[key] if np.isscalar(cq[key]) else cq[key][0]
                    for cq in cooling_quantities_list
                ]
            )
            for key in cooling_quantities_list[0].keys()
        }

        # ✅ Unpack simulation outputs
        g_s_values, F_values, T_mes_values, T_bs_values, T_soil_values, E_mes_values = (
            zip(*results)
        )

        # ✅ Compute A_mes and A_bs dependant on objective
        if obj_choice == "pump":
            A_mes = [
                pepc_assimilation_rate(T_mes_values[i], param_dict)
                for i in range(len(param_values))
            ]
            A_bs = [
                rubisco_assimilation_rate(T_bs_values[i], param_dict)
                for i in range(len(param_values))
            ]
        elif obj_choice == "nopump":
            A_mes = [0 for i in range(len(param_values))]
            A_bs = [
                rubisco_assimilation_rate(T_bs_values[i], param_dict)
                for i in range(len(param_values))
            ]
        else:
            A_bs = [0 for i in range(len(param_values))]
            A_mes = [
                rubisco_assimilation_rate(T_mes_values[i], param_dict)
                for i in range(len(param_values))
            ]

        # ✅ Compute total irradiance
        total_irradiance = compute_total_irradiance(
            param_dict=param_dict,
            vary_param=param_name,
            vary_values=param_values,
        )

        # ✅ Plot model results
        plot_all_graphs_A(
            parameter_name=param_name,
            parameter_values=param_values,
            g_s_values=list(g_s_values),
            A_mes_values=A_mes,
            A_bs_values=A_bs,
            T_mes_values=T_mes_values,
            T_bs_values=T_bs_values,
            E_mes_values=E_mes_values,
            F_opt_values=F_values,
            T_soil_values=T_soil_values,
            total_irradiance_values=total_irradiance,
            T_air_values=T_air_values if T_air_values is not None else None,
            param_dict=param_dict,
            param_unit=param_unit,
            error_T_mes_opt=error_T_mes_opt,
            error_T_bs_opt=error_T_bs_opt,
        )
        plot_summary(
            parameter_name=param_name,
            parameter_values=param_values,
            A_mes_values=A_mes,
            A_bs_values=A_bs,
            T_mes_values=T_mes_values,
            T_bs_values=T_bs_values,
            T_soil_values=T_soil_values,
            E_mes_values=E_mes_values,
            F_opt_values=F_values,
            param_dict=param_dict,
            param_unit=param_unit,
            error_T_mes_opt=error_T_mes_opt,
            error_T_bs_opt=error_T_bs_opt,
            nrows=nrows,
            ncols=ncols,
            objective=obj_choice,
        )

        # ✅ Plot cooling components
        plot_cooling_quantities_combined(
            parameter_name=param_name,
            parameter_values=param_values,
            cooling_quantities=combined_cooling_quantities,
            param_unit=param_unit,
            file_name_prefix=f"{file_prefix}_cooling_quantities",
            nrows=nrows,
            ncols=ncols,
        )

    return {
        "T_air": T_air_values if T_air_values is not None else None,
        "T_mes": list(T_mes_values),
        "T_bs": list(T_bs_values),
        "T_soil": list(T_soil_values),
        "E_mes": list(E_mes_values),
        "g_s": list(g_s_values),
        "F": list(F_values),
        "total_irradiance": total_irradiance.tolist(),
    }

In [132]:
def run_simulation_two_parameters(
    param1_name: str,
    param2_name: str,
    param1_values: np.ndarray,
    param2_values: np.ndarray,
    param_dict: Dict[str, float],
    error_T_mes_opt: float = 0.05,
    error_T_bs_opt: float = 0.05,
    elev: float = 30,
    azim: float = 60,
    param1_unit: str = r"(W \cdot m^{-2})",
    param2_unit: str = r"(W \cdot m^{-2})",
    file_prefix: str = "two_params",
    a: float = 0.0,
    b: float = 300.0,
    n_jobs: int = -1,
) -> Dict[str, np.ndarray]:
    """
    Runs a 2D parameter simulation over two variables and generates contour plots
    for 3-compartment model outputs and cooling quantities.
    """
    from joblib import Parallel, delayed
    import numpy as np
    import os

    def update_param_dict(p1: float, p2: float) -> Dict[str, float]:
        updated_dict = param_dict.copy()
        updated_dict[param1_name] = p1
        updated_dict[param2_name] = p2
        if param1_name == "phi":
            updated_dict["T_air"] = a * p1 + b
        elif param2_name == "phi":
            updated_dict["T_air"] = a * p2 + b
        return updated_dict

    shape = (len(param1_values), len(param2_values))

    # Dummy call to extract keys
    sample_param_dict = update_param_dict(param1_values[0], param2_values[0])
    g_msd, F_d, T_mes_d, T_bs_d, T_soil_d, _ = optimize_with_global_local(
        sample_param_dict
    )

    dummy_cooling = compute_cooling_quantities(
        g_m_values=np.array([g_s_d]),
        F_opt_values=np.array([F_d]),
        T_mes_values=np.array([T_mes_d]),
        T_bs_values=np.array([T_bs_d]),
        T_soil_values=np.array([T_soil_d]),
        param_dict=sample_param_dict,
    )

    # Initialize result grid
    combined = {
        key: np.zeros(shape)
        for key in [
            "T_air",
            "T_mes",
            "T_bs",
            "T_soil",
            "E_mes",
            "F",
            "g_s",
            "total_irradiance",
            *dummy_cooling.keys(),
        ]
    }

    def compute_at(i: int, j: int, p1: float, p2: float):
        pd = update_param_dict(p1, p2)
        g_s, F, T_mes, T_bs, T_soil, E_mes = optimize_with_global_local(pd)
        vary_param = param1_name if param1_name == "phi" else param2_name
        vary_value = pd[vary_param]
        total_irr = compute_total_irradiance(pd, vary_param, [vary_value])[0]
        cq = compute_cooling_quantities(
            g_s_values=np.array([g_s]),
            F_opt_values=np.array([F]),
            T_mes_values=np.array([T_mes]),
            T_bs_values=np.array([T_bs]),
            T_soil_values=np.array([T_soil]),
            param_dict=pd,
        )
        return (
            i,
            j,
            {
                "T_air": pd["T_air"],
                "g_s": g_s,
                "F": F,
                "T_mes": T_mes,
                "T_bs": T_bs,
                "T_soil": T_soil,
                "E_mes": E_mes,
                "total_irradiance": total_irr,
                **cq,
            },
        )

    results = Parallel(n_jobs=n_jobs)(
        delayed(compute_at)(i, j, p1, p2)
        for i, p1 in enumerate(param1_values)
        for j, p2 in enumerate(param2_values)
    )

    for i, j, computed in results:
        for key, value in computed.items():
            combined[key][i, j] = (
                value.item() if isinstance(value, np.ndarray) else value
            )

    # === Plot: Core Variables
    generate_all_contour_plots(
        param1_name=param1_name,
        param2_name=param2_name,
        param1_values=param1_values,
        param2_values=param2_values,
        T_mes_values=combined["T_mes"],
        T_bs_values=combined["T_bs"],
        T_soil_values=combined["T_soil"],
        E_mes_values=combined["E_mes"],
        F_values=combined["F"],
        g_s_values=combined["g_s"],
        cooling_quantities=combined,
        param_dict=param_dict,
        param1_unit=param1_unit,
        param2_unit=param2_unit,
    )

    # === Plot: Cooling Terms
    cooling_only = {
        k: v
        for k, v in combined.items()
        if k
        not in {
            "T_air",
            "T_mes",
            "T_bs",
            "T_soil",
            "E_mes",
            "F",
            "g_s",
            "total_irradiance",
        }
    }

    generate_contour_plot_cooling_quantities_combined(
        param1_name=param1_name,
        param2_name=param2_name,
        param1_values=param1_values,
        param2_values=param2_values,
        cooling_quantities=cooling_only,
        param_unit1=param1_unit,
        param_unit2=param2_unit,
        file_name_prefix=f"{file_prefix}_cooling_quantities_combined",
    )

    return combined

In [133]:
def plot_pepc_vs_HCO3(param_dict: Dict[str, float]) -> None:
    """
    Beautiful Beamer-ready plot of PEPC assimilation rate vs mesophyll temperature.
    """
    sns.set_context("notebook", font_scale=1.4)
    A_values = []
    dA_dHCO3_vals = []
    T_fixed = Param_value(273.15 + 35, "degK")
    C_values = np.linspace(0, 200, 1000)  # CO2 concentration in Pa
    for CO2_values in C_values:
        p_copy = param_dict.copy()
        p_copy["bicarbonate"] = (CO2_values, "Pa")

        A_values.append(pepc_assimilation_rate(T_fixed, p_copy))
        dA = dpepc_rate(T_fixed, p_copy)
        dA_dHCO3_vals.append(dA)
    A_max = max(A_values)
    dA_values_nodimm = [dA.value for dA in dA_dHCO3_vals]
    A_values_nodimm = [A.value for A in A_values]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(
        C_values,
        A_values_nodimm,
        color="black",
        linewidth=4.5,
        label="PEPC Assimilation",
    )
    ax.plot(
        C_values,
        dA_values_nodimm,
        color="blue",
        linestyle="--",
        linewidth=2.5,
        label="dA/dHCO₃",
    )
    # ax.axvline(
    #     T_opt,
    #     color="red",
    #     linestyle="--",
    #     linewidth=1.5,
    #     label=f"Optimum = {T_opt:.2f} K"
    # )

    ax.set_xlabel(r"$CO_2$ air concentration [Pa]")
    ax.set_ylabel("PEPC rate [mol bic m⁻² s⁻¹]")
    ax.grid(True, linestyle=":", alpha=0.5)
    # ax.legend(loc="upper left")

    sns.despine()
    plt.tight_layout()

    # Ensure the folder exists
    os.makedirs("figures", exist_ok=True)
    plt.savefig("figures/pepc_temperature_response.png", dpi=300, transparent=True)

    plt.show()
    # print(f"Maximum PEPC rate = {A_max:.3e}")


def plot_rubisco_vs_temperature(param_dict: Dict[str, float]) -> None:
    """
    Beautiful Beamer-ready plot of Rubisco assimilation rate vs bundle sheath temperature.
    """
    sns.set_context("notebook", font_scale=1.4)

    T_values = np.linspace(250.15, 350, 1000)
    A_rubisco = [rubisco_assimilation_rate(T, param_dict) for T in T_values]
    A_max = max(A_rubisco)
    T_opt = T_values[np.argmax(A_rubisco)]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(
        T_values, A_rubisco, color="black", linewidth=4.5, label="Rubisco Assimilation"
    )
    ax.axvline(
        T_opt,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label=f"Optimum = {T_opt:.2f} K",
    )

    ax.set_xlabel(r"$T_{bs}$ $[K]$")
    ax.set_ylabel("Rubisco rate [mol CO₂ m⁻² s⁻¹]")
    ax.grid(True, linestyle=":", alpha=0.5)
    # ax.legend(loc="upper left")

    sns.despine()
    plt.tight_layout()

    # Ensure the output directory exists
    os.makedirs("figures", exist_ok=True)
    plt.savefig(
        "figures/rubisco_vs_temperature.png",
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )

    plt.show()
    print(f"Maximum Rubisco rate = {A_max:.3e} at {T_opt:.2f} K")

In [134]:
param_dict = {
    "C_w": (4184.0, "J/(kg*degK)"),
    "khi": (2450000.0, "J/kg"),
    "R": (8.314, "J/(mol*degK)"),
    "T_air": (308.15, "degK"),
    "T_sink": (283.15, "degK"),
    "alpha_mes": (0.5, None),
    "alpha_bs": (0.6, None),
    "alpha_surf": (0.7, None),
    "alpha_root": (0.0, None),
    "tau_mes": (0.4, None),
    "tau_bs": (0.4, None),
    "tau_surf": (0.0, None),
    "phi": (1000, "W/m**2"),
    "gamma_mes": (0.3, "W/(m**2*degK)"),
    "gamma_bs": (0.3, "W/(m**2*degK)"),
    "gamma_soil": (0.4, "W/(m**2*degK)"),
    "h_mes": (10.0, "W/(m**2*degK)"),
    "h_soil": (60.0, "W/(m**2*degK)"),
    "k_soil": (0.5, "W/(m**2*degK)"),
    "d_soil": (0.5, None),  # dimension error, WIP
    "U_mes_bs": (50, "W/(m**2*degK)"),
    "U_surf_root": (25, "W/(m**2*degK)"),
    "U_root_sink": (20, "W/(m**2*degK)"),
    "E_surf": (2e-05, "kg/s"),
    "rubisco": (1e-06, "mol/m**3"),
    "PEPC": (2e-06, "mol/m**3"),
    "Vpmax_25": (0.0001, "mol/(m**2*s)"),  # Consistent with other terms in PEPC
    "Ea_Vpmax": (65000, "J/mol"),
    "DS_Vpmax": (666, "J/(mol*degK)"),
    "Hd_Vpmax": (210000, "J/mol"),
    "K_HCO3_25": (26, "mol/m**3"),
    "Ea_K_HCO3": (36300, "J/mol"),
    "epsilon_lower": (0.0006, None),
    "epsilon_upper": (0.006, None),
    "E_mes_BOUND": ((0, 1), "kg/(m**2*s)"),
    "F_BOUND": ((0, 0.002), "kg/(m**2*s)"),
    "a_value": (0.01, None),  # WIP : add back units ("degK*m**2)/W")
    "b_value": (289.0, None),  # WIP : add back units ("degK")
    "Kp_25": (2.2e-05, "mol/m**3"),
    "weight_rub": (0.8, None),
    "weight_pepc": (0.2, None),
    "CO2_air": (42.56, "Pa"),
    "CO2_mes_init": (30, "Pa"),  # Unused, but kept for consistency
    "CO2_bs_init": (30, "Pa"),
    "O2_air": (21000, "Pa"),
    "O2_bs_init": (0.02, "Pa"),  # 1e-6 smaller than O2_air
    "g_m": (1e-05, "mol/(m**2*s*Pa)"),
    "Vcmax_25": (6e-05, "mol/(m**2*s)"),
    "Ea_Vcmax": (78000, "J/mol"),
    "DS_Vcmax": (650, "J/(mol*degK)"),
    "Hd_Vcmax": (200000, "J/mol"),
    "Kc_25": (121, "mol/m**3"),
    "Ea_Kc": (64200, "J/mol"),
    "Ko_25": (29200, "mol/m**3"),
    "Ea_Ko": (10500, "J/mol"),
    "Sco_25": (1310, "mol/mol"),
    "Ea_Sco": (-31100, "J/mol"),
    "CA": (0.01, "mol/m**3"),
    "kcat_CA": (100.0, "mol/(m**2*s)"),
    "K_CA": (0.001, "mol/m**3"),
    "g_bs": (1.64e-08, "mol/(m**2*s*Pa)"),
    "A_mes": (2.5, "m**2/m**2"),  # Mesophyll surface area per m² soil surface
    "A_bs": (1.5, "m**2/m**2"),  # Bundle sheath area per m² soil surface
    "A_root": (1.0, "m**2/m**2"),  # Approximate root surface area per m² soil
    "A_contact": (
        0.5,
        "m**2/m**2",
    ),  # ~50% of mesophyll-BS area assumed effectively exchanging heat
    "zeta": (0.85, "m**2/m**2"),  # Leaf coverage fraction (dense but not closed canopy)
    "A_circ": (0.6, "m**2/m**2"),  # Optional, if needed directly
    "g_s_BOUND": (
        (1e-08, 1e-05),
        "mol/(m**2*s*Pa)",
    ),  # Boundaries for g_s in mol m⁻² s⁻¹ Pa⁻¹
    "RH": (0.8, None),  # Relative humidity
    "T_mes_opt": (313.15, "degK"),  # Optimal mesophyll temperature in Kelvin
    "T_bs_opt": (306.15, "degK"),  # Optimal bundle sheath temperature in Kelvin
}

In [135]:
# plot_pepc_vs_HCO3(param_dict)

In [136]:
DEFAULT_CONSTANTS = {
    "M_water": (0.01801528, "kg/mol"),  # kg/mol (molar mass of water)
    "P_atm": (101325.0, "Pa"),  # Pa (standard atmospheric pressure)
    "kelvin_to_celsius": (273.15, "degK"),  # K to °C offset
    "tetens_a": (0.6108, "kPa"),  # kPa (Tetens equation coefficient)
    "tetens_b": (17.27, None),  # Tetens equation coefficient
    "tetens_c": (237.3, None),  # °C (Tetens equation coefficient)
    "kPa_to_Pa": 1000.0,  # Conversion from kPa to Pa necessary for Tetens equation because we can not use Pint for it.
    "R": (8.314, "J/(mol*degK)"),  # J/(mol·K) (Universal gas constant)
}

# Example of applying this to a parameter dictionary
param_dict["constants"] = DEFAULT_CONSTANTS

In [137]:
new_params = {
    # 🌧️ Water Availability
    "rainfall_rate": (2.31e-5, "kg/(s*m**2)"),  # kg/s·m², ~2 mm/day
    "A_leaf": (3.0, None),  # Leaf area index (dimensionless)
    # VPD & Water Vapor (in mol/m³)
    "VPD_threshold": (
        4000.0,
        "Pa",
    ),  # Pa → will be converted to mol/m³ internally WIP : Implement rain-based transpiration constraint
    # 🌡️ Thermal Penalty Threshold
    "T_max": (318.15, "degK"),  # K (soft upper failure limit)
    # ⚖️ Penalty Weights
    "lambda_water": (10.0, None),
    "lambda_vpd": (5.0, None),
    "lambda_heat": (0.2, None),
    # ⚡ Light-dependent Electron Transport
    "J_max25": (160e-6, "mol/(m**2*s)"),  # mol/m²/s, typical value for C₄
    "Ea_J": (37830.0, "J/mol"),  # J/mol
    "DS_J": (644.4338, "J/mol"),  # J/mol/K
    "Hd_J": (200000.0, "J/mol"),  # J/mol
    "theta": (0.7, None),  # non-rectangular curvature
    #  Compensation point (optional)
    "Gamma_star": (4e-6, "mol/mol"),  # mol/mol (unitless)
    # default values for carbon, oxygen, and bicarbonate concentrations
    "carbon": (
        40,
        "Pa",
    ),  # Pa (carbon dioxide concentration) WIP : Check if unit is correct
    "oxygen": (21278, "Pa"),  # Pa (oxygen concentration)
    "bicarbonate": (0.001, "Pa"),
}


param_dict.update(new_params)

In [138]:
param_dict

{'C_w': (4184.0, 'J/(kg*degK)'),
 'khi': (2450000.0, 'J/kg'),
 'R': (8.314, 'J/(mol*degK)'),
 'T_air': (308.15, 'degK'),
 'T_sink': (283.15, 'degK'),
 'alpha_mes': (0.5, None),
 'alpha_bs': (0.6, None),
 'alpha_surf': (0.7, None),
 'alpha_root': (0.0, None),
 'tau_mes': (0.4, None),
 'tau_bs': (0.4, None),
 'tau_surf': (0.0, None),
 'phi': (1000, 'W/m**2'),
 'gamma_mes': (0.3, 'W/(m**2*degK)'),
 'gamma_bs': (0.3, 'W/(m**2*degK)'),
 'gamma_soil': (0.4, 'W/(m**2*degK)'),
 'h_mes': (10.0, 'W/(m**2*degK)'),
 'h_soil': (60.0, 'W/(m**2*degK)'),
 'k_soil': (0.5, 'W/(m**2*degK)'),
 'd_soil': (0.5, None),
 'U_mes_bs': (50, 'W/(m**2*degK)'),
 'U_surf_root': (25, 'W/(m**2*degK)'),
 'U_root_sink': (20, 'W/(m**2*degK)'),
 'E_surf': (2e-05, 'kg/s'),
 'rubisco': (1e-06, 'mol/m**3'),
 'PEPC': (2e-06, 'mol/m**3'),
 'Vpmax_25': (0.0001, 'mol/(m**2*s)'),
 'Ea_Vpmax': (65000, 'J/mol'),
 'DS_Vpmax': (666, 'J/(mol*degK)'),
 'Hd_Vpmax': (210000, 'J/mol'),
 'K_HCO3_25': (26, 'mol/m**3'),
 'Ea_K_HCO3': (36300,

In [139]:
def plot_rubisco_vs_co2(param_dict: Dict[str, float]) -> None:
    """
    Beautiful Beamer-ready plot of Rubisco assimilation rate vs CO2 concentration.
    """
    sns.set_context("notebook", font_scale=1.4)

    Air_CO2 = np.linspace(0.01, 0.1, 150)  # CO2 in Pa

In [140]:
# plot rubisco assimilation rate vs temperature

# plot_rubisco_vs_temperature(param_dict)

In [141]:
# plot pepc assimilation rate vs temperature
# plot_pepc_vs_temperature(param_dict)

In [142]:
# # plot air temperature
# param_name = "phi"
# param_values = np.linspace(400, 1200, 50)
# param_dict["phi"] = param_values
# param_dict["T_air"] = param_dict["a_value"] * param_values + param_dict["b_value"]
# plot_total_irradiance(
#     vary_values=param_values,
#     total_irradiance=param_dict["T_air"],
#     vary_param=param_name,
#     param_unit=r"(W \cdot m^{-2})",
#     file_name_prefix="air_temperature",
# )

In [143]:
# phi_values = np.array([Param_value(p, "W/m**2") for p in np.linspace(400, 1000, 30)])   # Vary solar radiation from 400 to 900 W/m²
phi_values = Param_value.array_from_quantities(
    np.linspace(400, 1000, 30), "W*m**2"
)  # Vary solar radiation from 400 to 1000 W/m²

phi_values[0]

400.0 m²·W

In [144]:
# Define parameter bounds for optimization
phi_values = np.linspace(
    400, 1000, 30
)  # Vary solar radiation from 400 to 1000 W/m² dimensionality error with a, b and phi WIP
phi_dim = "W*m**2"  # Dimension for solar radiation
# CO2_air_values = np.linspace(0, 150, 30) # Vary CO2 concentration from 0 to 0.086 mol/m³
# Run single parameter simulation for varying phi (solar radiation)
simulation_results_irradiance = run_simulation_one_parameter_two_conditions(
    param_name="phi",
    param_values=phi_values,
    param_dim=phi_dim,
    param_dict=param_dict,
    error_T_mes_opt=0.001,  # Tight tolerance for T_mes optimization
    error_T_bs_opt=0.001,  # Tight tolerance for T_bs optimization
    param_unit=r"(mol \cdot m^{-3})",
    file_prefix="air_CO2_variation",
    a=param_dict["a_value"][
        0
    ],  # Coefficient for T_air = a*phi + b Dimensionality error with a, b and phi WIP
    b=param_dict["b_value"][0],  # Intercept for T_air
)

# Display the results
print("Simulation complete. Results stored in `simulation_results` dictionary.")

DimensionalityError: Cannot convert from 'watt * meter ** 2' ([mass] * [length] ** 4 / [time] ** 3) to 'watt / meter ** 2 / kelvin' ([mass] / [time] ** 3 / [temperature])

In [ ]:
# plot in subplots(1,2) irradiance vs  pepc and rubisco using the results

In [ ]:
# Define parameter bounds for optimization
RH_values = np.linspace(0, 1, 30)  # Vary solar radiation from 400 to 900 W/m²

# Run single parameter simulation for varying phi (solar radiation)
simulation_results_irradiance = run_simulation_one_parameter(
    param_name="phi",
    param_values=RH_values,
    param_dict=param_dict,
    error_T_mes_opt=0.001,  # Tight tolerance for T_mes optimization
    error_T_bs_opt=0.001,  # Tight tolerance for T_bs optimization
    param_unit=r"(non-dimensional)",
    file_prefix="RH_variation",
    a=param_dict["a_value"],  # Coefficient for T_air = a*phi + b
    b=param_dict["b_value"],  # Intercept for T_air
)

# Display the results
print("Simulation complete. Results stored in `simulation_results` dictionary.")

In [ ]:
# Define parameter bounds for optimization
zeta_values = np.linspace(1e-8, 1, 30)  # Vary solar radiation from 400 to 900 W/m²

# Run single parameter simulation for varying phi (solar radiation)
simulation_results_zeta = run_simulation_one_parameter(
    param_name="zeta",
    param_values=zeta_values,
    param_dict=param_dict,
    error_T_mes_opt=0.001,  # Tight tolerance for T_mes optimization
    error_T_bs_opt=0.001,  # Tight tolerance for T_bs optimization
    param_unit=r"(unitless)",
    file_prefix="zeta_variation",
    a=param_dict["a_value"],  # Coefficient for T_air = a*phi + b
    b=param_dict["b_value"],  # Intercept for T_air
)

# Display the results
print("Simulation complete. Results stored in `simulation_results` dictionary.")

In [ ]:
# Define ranges for two parameters
phi_values = np.linspace(400, 1200, 10)
zeta_values = np.linspace(0.1, 0.9, 10)

# Run two parameters simulation
run_simulation_two_parameters(
    param1_name="phi",
    param2_name="zeta",
    param1_values=phi_values,
    param2_values=zeta_values,
    param_dict=param_dict,
    # error_T_mes_opt=0.05,
    # error_T_bs_opt=0.05,
    param1_unit=r"(W \cdot m^{-2})",
    param2_unit=r"(unitless)",
    elev=30,  # Elevation angle for 3D plots
    azim=60,  # Azimuthal angle for 3D plots
    a=param_dict["a_value"],  # Coefficient for T_air = a*phi + b
    b=param_dict["b_value"],  # Intercept for T_air
)

# 🌱 Energy Balance Model Simulation

## 📌 Overview

This repository contains a **Python-based energy balance model** that simulates the **temperature, transpiration, and irradiance dynamics** in a system based on four steady-state temperature balance equations.

The model incorporates:

- **Heat balance equations** to compute temperatures of the mesophyll, bundle sheath, soil surface, soil root area.
- **Optimization routines** to maximize assimilation rates of PEPC and Rubisco as a function of temperature of the mesophyll and bundle sheath, respectively.
- **Enzyme kinetics** to model the temperature dependence of PEPC and Rubisco.
- **Energy conservation checks** to ensure that the energy balance is maintained.
- **Customizable model parameters** to easily modify physical constants, enzyme kinetics, and bounds.
- **Parallel computation** to support multi-threaded optimization via `joblib`.
- **Irradiance calculations** to ensure energy conservation.
- **Dynamic visualizations** for easy analysis.

## 🚀 Features

✅ **Numerical Optimization** - Uses `scipy.optimize` (dual annealing + L-BFGS-B).  
✅ **Energy Conservation Checks** - Computes **radiative, convective, and conductive heat flows**.  
✅ **Customizable Model Parameters** - Easily modify physical constants, enzyme kinetics, and bounds.  
✅ **Parallel Computation** - Supports multi-threaded optimization via `joblib`.  
✅ **Dynamic Visualization** - Generates **line plots, contour plots, and 3D surface plots**.

---

## 📂Function Dependency Graph

graph TD;
A[run_simulation_one_parameter] -->|calls| B[optimize_with_global_local]
B -->|calls| C[find_stable_initial_guess]
C -->|calls| D[dual_annealing]
C -->|calls| E[objective]
E -->|calls| F[calculate_temperatures]
F -->|calls| G[compute_total_irradiance]
F -->|calls| H[rubisco_assimilation_rate]
F -->|calls| I[pepc_assimilation_rate]
E -->|calls| B[optimize_with_global_local]
E -->|calls| J[scipy_minimize (L-BFGS-B)]
B -->|calls| K[compute_cooling_quantities]
B -->|calls| L[plot_all_graphs]
B -->|calls| M[plot_total_irradiance]
B -->|calls| N[plot_cooling_quantities_combined]
B -->|returns| O[simulation results]


In [ ]:
"""
run_simulation_one_parameter()
│
├── 🟠 optimize_with_global_local()
│   ├── 🟠 find_stable_initial_guess()
│   │   ├── 🟠 dual_annealing()
│   │   ├── 🔵 objective()
│   │   │   ├── 🔵 calculate_temperatures()
│   │   │   │   ├── 🔵 compute_total_irradiance()
│   │   │   │   ├── 🔵 rubisco_assimilation_rate()
│   │   │   │   ├── 🔵 pepc_assimilation_rate()
│   │   │   ├── 🟠 optimize_with_global_local()
│   │   │   └── 🟠 scipy_minimize()  (L-BFGS-B optimization)
│   │   ├── 🟢 compute_cooling_quantities()
│   │   ├── 🟣 plot_all_graphs()
│   │   ├── 🟣 plot_total_irradiance()
│   │   ├── 🟣 plot_cooling_quantities_combined()
│   │   └── 🟢 return simulation results

"""

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import os
from scipy.optimize import minimize


def plot_objective_landscape_simple(
    param_dict, objective_fn, n_points=100, E_bounds=None, F_bounds=None
):
    """
    Plot optimization landscape for a given objective function over (E_mes, F),
    with the optimal point marked.

    Args:
        param_dict (dict): Dictionary of model parameters.
        objective_fn (function): Function to evaluate objective.
        n_points (int): Number of grid points per dimension.
        E_bounds (tuple): Tuple of (E_min, E_max) if provided.
        F_bounds (tuple): Tuple of (F_min, F_max) if provided.
    """
    sns.set_style("whitegrid")
    rcParams.update(
        {
            "font.size": 16,
            "axes.labelsize": 18,
            "axes.titlesize": 20,
            "xtick.labelsize": 14,
            "ytick.labelsize": 14,
            "legend.fontsize": 14,
            "figure.titlesize": 20,
            "axes.formatter.useoffset": False,
        }
    )

    if E_bounds is None:
        E_min, E_max = param_dict["E_mes_BOUND"]
    else:
        E_min, E_max = E_bounds

    if F_bounds is None:
        F_min, F_max = param_dict["F_BOUND"]
    else:
        F_min, F_max = F_bounds

    E_vals = np.linspace(E_min, E_max, n_points)
    F_vals = np.linspace(F_min, F_max, n_points)
    E_grid, F_grid = np.meshgrid(E_vals, F_vals)

    Z = np.full_like(E_grid, np.nan)

    for i in range(n_points):
        for j in range(n_points):
            E_ij = E_grid[i, j]
            F_ij = F_grid[i, j]
            Z[i, j] = -objective_fn((E_ij, F_ij), param_dict)

    # Compute optimal point using minimize
    result = minimize(
        objective_fn,
        [np.mean(E_vals), np.mean(F_vals)],
        args=(param_dict,),
        bounds=[(E_min, E_max), (F_min, F_max)],
    )
    E_opt, F_opt = result.x if result.success else (np.nan, np.nan)

    fig, ax = plt.subplots(figsize=(10, 8))
    contour = ax.contourf(E_grid, F_grid, Z, cmap="viridis", levels=50)
    ax.set_title("Objective Function Landscape", fontsize=18, weight="bold")
    ax.set_xlabel("Eₘₑₛ [kg m⁻² s⁻¹]")
    ax.set_ylabel("F [kg m⁻² s⁻¹]")
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0e}"))
    ax.tick_params(axis="x", rotation=45)
    ax.plot(E_opt, F_opt, "rX", markersize=10, label="Optimal Point")
    ax.legend()
    cbar = fig.colorbar(contour, ax=ax, orientation="horizontal", pad=0.2, shrink=0.9)
    cbar.set_label("Objective Value", fontsize=14)

    os.makedirs("figure_presentation_objective", exist_ok=True)
    fig.savefig(
        "figure_presentation_objective/simple_objective_landscape.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# Call the plotting function
plot_objective_landscape_simple(
    param_dict, objective, n_points=100, E_bounds=(0, 2e-4), F_bounds=(0, 1e-2)
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from typing import Tuple, Dict


def plot_objective_vs_E_F_with_optimal(
    objective: callable,
    param_dict: Dict[str, float],
    E_bounds: Tuple[float, float],
    F_bounds: Tuple[float, float],
    n_points: int = 100,
    save_path: str = "objective_landscape_with_optimal.png",
) -> None:
    """
    Plots objective vs E and F as a contour plot with optimal point overlay.
    """
    E_vals = np.linspace(E_bounds[0], E_bounds[1], n_points)
    F_vals = np.linspace(F_bounds[0], F_bounds[1], n_points)
    E_grid, F_grid = np.meshgrid(E_vals, F_vals)

    # Compute objective values
    Z = np.array([[-objective((e, f), param_dict) for e in E_vals] for f in F_vals])

    # Find optimal point using local optimization
    bounds = [E_bounds, F_bounds]
    initial_guess = [np.mean(E_bounds), np.mean(F_bounds)]
    result = minimize(objective, initial_guess, args=(param_dict,), bounds=bounds)

    if not result.success:
        raise RuntimeError("Optimization failed")

    E_opt, F_opt = result.x
    obj_opt = -result.fun

    # Plotting
    plt.figure(figsize=(10, 8))
    contour = plt.contourf(E_grid, F_grid, Z, levels=50, cmap="viridis")
    plt.plot(
        E_opt,
        F_opt,
        "rX",
        markersize=10,
        label=f"Opt: E={E_opt:.1e}, F={F_opt:.1e}, Obj={obj_opt:.1e}",
    )
    plt.colorbar(contour)
    plt.xlabel("E_mes [kg m⁻² s⁻¹]")
    plt.ylabel("F [kg m⁻² s⁻¹]")
    plt.xticks(rotation=45, fontsize=12)
    plt.yticks(fontsize=12)
    # set major ticks to scientific notation and rotate x-axis labels
    plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1e}"))
    plt.gca().tick_params(axis="x", rotation=45)
    plt.gca().tick_params(axis="y", rotation=0)

    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()

    print(f"✅ Optimal E = {E_opt:.2e}, F = {F_opt:.2e}, Objective = {obj_opt:.3e}")
    print(f"📁 Plot saved to: {save_path}")

In [ ]:
plot_objective_vs_E_F_with_optimal(
    objective, param_dict, E_bounds=(0, 2e-4), F_bounds=(0, 1e-2), n_points=100
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.ticker import FuncFormatter
import seaborn as sns


def plot_objective_landscape_epsilon(param_dict, objective, n_points=100):
    """
    Fancy Beamer-optimized plot of Rubisco-only objective under epsilon-constraint.
    This version plots the landscape of the provided objective function (which is defined as:

        objective = -1e3 * scale_factor * A_rubisco

    so that minimizing the objective corresponds to maximizing Rubisco assimilation rate).
    The optimum (i.e. minimum objective) is marked and annotated with its corresponding
    Rubisco assimilation rate.

    It is assumed that the following functions are defined elsewhere:
      - calculate_temperatures(F, E_mes, param_dict)

    The parameter dictionary is used without modification.
    """
    sns.set_style("whitegrid")
    rcParams.update(
        {
            "font.size": 16,
            "axes.labelsize": 18,
            "axes.titlesize": 20,
            "xtick.labelsize": 14,
            "ytick.labelsize": 14,
            "legend.fontsize": 14,
            "figure.titlesize": 20,
            "axes.formatter.useoffset": False,
        }
    )

    # Use provided bounds.
    E_min, E_max = param_dict["E_mes_BOUND"]
    F_min, F_max = param_dict["F_BOUND"]
    E_vals = np.linspace(E_min, E_max, n_points)
    F_vals = np.linspace(F_min, F_max, n_points)
    E_grid, F_grid = np.meshgrid(E_vals, F_vals)

    # We'll compute the objective at each grid point.
    # If the objective function returns the penalty value 1e6, we mark it as infeasible.
    Z = np.full_like(E_grid, np.nan, dtype=float)
    mask = np.full_like(E_grid, False, dtype=bool)
    for i in range(n_points):
        for j in range(n_points):
            E_ij, F_ij = E_grid[i, j], F_grid[i, j]
            try:
                val = -objective((E_ij, F_ij), param_dict)
                Z[i, j] = val
                if np.isclose(val, 1e6):
                    mask[i, j] = True
            except Exception:
                mask[i, j] = True

    # Mask out infeasible zones.
    Z_masked = np.ma.masked_where(mask, Z)

    # Find the optimum: since the objective is negative for feasible points,
    # the minimum value corresponds to the maximum Rubisco assimilation rate.
    min_idx = np.unravel_index(np.nanargmax(Z), Z.shape)
    E_opt = E_grid[min_idx]
    F_opt = F_grid[min_idx]

    # Retrieve temperatures at the optimum.
    T_mes_opt, T_bs_opt, _, _ = calculate_temperatures(F_opt, E_opt, param_dict)

    # Compute the optimum Rubisco assimilation rate.
    # Given: objective = -1e3 * scale_factor * A_rubisco.
    # Here, scale_factor is 1e6 as defined in your objective.
    scale_factor = 1e6
    optimal_objective = Z[min_idx]
    A_rub_opt = optimal_objective

    fig, ax1 = plt.subplots(figsize=(10, 8))
    contour = ax1.contourf(
        E_grid, F_grid, Z_masked, cmap="viridis", levels=50, alpha=0.95
    )
    # Optionally, shade infeasible zones.
    ax1.contourf(E_grid, F_grid, mask, levels=[0.5, 1.5], colors=["#ffcccc"], alpha=0.4)

    # Mark the optimum.
    ax1.plot(E_opt, F_opt, "rX", markersize=12, label="Optimized")
    ax1.legend(loc="upper right")
    # ax1.set_title("Rubisco Assimilation Objective Landscape", fontsize=18, weight="bold")
    ax1.set_xlabel("Eₘₑₛ [kg m⁻² s⁻¹]")
    ax1.set_ylabel("F [kg m⁻² s⁻¹]")
    ax1.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0e}"))
    ax1.tick_params(axis="x", rotation=45)
    cbar = fig.colorbar(contour, ax=ax1, orientation="vertical", pad=0.2, shrink=0.9)
    cbar.set_label("Objective Value", fontsize=14)

    # annotation_text = (
    #     f"Optimum:\n"
    #     f"Eₘₑₛ = {E_opt:.2e}\n"
    #     f"F = {F_opt:.2e}\n"
    #     f"Tₘₑₛ = {T_mes_opt:.1f} K\n"
    #     f"T_bs = {T_bs_opt:.1f} K\n"
    #     f"Aᵣᵤb = {A_rub_opt:.2e} mol CO₂ m⁻² s⁻¹"
    # )
    # ax1.annotate(annotation_text, xy=(E_opt, F_opt), xytext=(20, 20), textcoords="offset points",
    #              arrowprops=dict(arrowstyle="->", color='red'),
    #              fontsize=12, color='red', weight="bold")

    fig.suptitle(
        f"Optimum → Eₘₑₛ={E_opt:.2e}, F={F_opt:.2e} | "
        f"Tₘₑₛ={T_mes_opt:.1f}K, T_bs={T_bs_opt:.1f}K | "
        f"Aᵣᵤb={A_rub_opt:.2e} mol CO₂ m⁻² s⁻¹ | "
        f"ϕ={param_dict['phi']} W/m²",
        fontsize=16,
        y=0.99,
    )

    os.makedirs("figure_presentation_epsilon", exist_ok=True)
    fig.savefig(
        "figure_presentation_epsilon/objective_landscape_bio_full.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# param_dict["E_mes_BOUND"] = (0, 2e-3)
# param_dict["F_BOUND"] = (0, 3e-2)
# param_dict["phi"] = 800

# plot_objective_landscape_epsilon(param_dict, objective, n_points=100)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.ticker import ScalarFormatter
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import seaborn as sns


def plot_objective_landscape_epsilon_3d(
    param_dict: dict, objective: callable, n_points: int = 100
) -> None:
    """
    Create a high-quality 3D surface plot of the Rubisco objective landscape
    under an epsilon-constraint using the provided objective function.

    The plot highlights the optimum point and masks infeasible regions.
    """
    sns.set_style("whitegrid")
    rcParams.update(
        {
            "font.size": 14,
            "axes.labelsize": 16,
            "axes.titlesize": 18,
            "xtick.labelsize": 12,
            "ytick.labelsize": 12,
            "legend.fontsize": 12,
            "figure.titlesize": 18,
        }
    )

    E_min, E_max = param_dict["E_mes_BOUND"]
    F_min, F_max = param_dict["F_BOUND"]
    E_vals = np.linspace(E_min, E_max, n_points)
    F_vals = np.linspace(F_min, F_max, n_points)
    E_grid, F_grid = np.meshgrid(E_vals, F_vals)

    Z = np.full_like(E_grid, np.nan, dtype=float)
    mask = np.full_like(E_grid, False, dtype=bool)

    for i in range(n_points):
        for j in range(n_points):
            E_ij, F_ij = E_grid[i, j], F_grid[i, j]
            try:
                val = -objective((E_ij, F_ij), param_dict)
                Z[i, j] = val
                if np.isclose(val, 1e6):
                    mask[i, j] = True
            except Exception:
                mask[i, j] = True

    Z_masked = np.ma.masked_where(mask, Z)

    min_idx = np.unravel_index(np.nanargmax(Z), Z.shape)
    E_opt = E_grid[min_idx]
    F_opt = F_grid[min_idx]
    T_mes_opt, T_bs_opt, _, _ = calculate_temperatures(F_opt, E_opt, param_dict)
    A_rub_opt = Z[min_idx]

    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection="3d")

    surf = ax.plot_surface(
        E_grid,
        F_grid,
        Z_masked,
        cmap="viridis",
        edgecolor="k",
        linewidth=0.1,
        alpha=0.95,
        antialiased=True,
    )

    ax.scatter(
        E_opt, F_opt, A_rub_opt, color="red", s=100, label="Optimum", depthshade=True
    )

    # ax.set_title("Rubisco Objective Landscape (3D)", pad=15, weight="bold")
    ax.set_xlabel("Eₘₑₛ [kg m⁻² s⁻¹]", labelpad=10)
    ax.set_ylabel("F [kg m⁻² s⁻¹]", labelpad=10)
    ax.set_zlabel("Objective Value", labelpad=10)

    ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))

    ax.ticklabel_format(style="sci", axis="x", scilimits=(0, 0))
    ax.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
    ax.ticklabel_format(style="sci", axis="z", scilimits=(0, 0))

    ax.view_init(elev=38, azim=155)  # Adjust the view angle for better visibility
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.1)

    ax.legend(loc="best")

    fig.subplots_adjust(top=0.85)  # Give room for suptitle
    fig.suptitle(
        f"Eₘₑₛ={E_opt:.2e}, F={F_opt:.2e} | "
        f"Tₘₑₛ={T_mes_opt:.1f}K, T_bs={T_bs_opt:.1f}K | "
        f"Aᵣᵤb={A_rub_opt:.2e} mol CO₂ m⁻² s⁻¹ | "
        f"ϕ={param_dict['phi']} W/m²",
        fontsize=13,
    )

    os.makedirs("figure_presentation_epsilon", exist_ok=True)
    fig.savefig(
        "figure_presentation_epsilon/objective_landscape_bio_full_3d.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.tight_layout()
    plt.show()

In [ ]:
plot_objective_landscape_epsilon_3d(param_dict, objective, n_points=100)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", context="notebook")


def compute_pepc_rubisco_rates(
    T_range: np.ndarray, param_dict: dict
) -> tuple[np.ndarray, np.ndarray]:
    """Compute PEPC and Rubisco rates across temperatures."""
    R = param_dict["R"]
    Vpmax, Vcmax = [], []

    for T in T_range:
        # PEPC
        Vpmax_T = (
            param_dict["Vpmax_25"]
            * np.exp(param_dict["Ea_Vpmax"] * (T - 298.15) / (298.15 * R * T))
            * (
                1
                + np.exp(
                    (298.15 * param_dict["DS_Vpmax"] - param_dict["Hd_Vpmax"])
                    / (298.15 * R)
                )
            )
            / (
                1
                + np.exp(
                    (T * param_dict["DS_Vpmax"] - param_dict["Hd_Vpmax"]) / (T * R)
                )
            )
        )
        Vpmax.append(Vpmax_T)

        # Rubisco
        Vcmax_T = (
            param_dict["Vcmax_25"]
            * np.exp(param_dict["Ea_Vcmax"] * (T - 298.15) / (298.15 * R * T))
            * (
                1
                + np.exp(
                    (298.15 * param_dict["DS_Vcmax"] - param_dict["Hd_Vcmax"])
                    / (298.15 * R)
                )
            )
            / (
                1
                + np.exp(
                    (T * param_dict["DS_Vcmax"] - param_dict["Hd_Vcmax"]) / (T * R)
                )
            )
        )
        Vcmax.append(Vcmax_T)

    return np.array(Vpmax), np.array(Vcmax)


def plot_assimilation_subplots(
    T_range: np.ndarray,
    param_dict: dict,
    title_pepc: str = "PEPC Assimilation vs Temperature",
    title_rubisco: str = "Rubisco Assimilation vs Temperature",
) -> None:
    """Plot PEPC and Rubisco assimilation responses with user-defined temperature range (in Kelvin)."""
    Vpmax, Vcmax = compute_pepc_rubisco_rates(T_range, param_dict)

    T_opt_pepc = T_range[np.argmax(Vpmax)]
    T_opt_rubisco = T_range[np.argmax(Vcmax)]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    # PEPC plot
    ax = axes[0]
    ax.plot(T_range, Vpmax, color="seagreen", linewidth=2.5, label="PEPC")
    ax.axvline(
        T_opt_pepc,
        color="seagreen",
        linestyle="--",
        alpha=0.7,
        label=f"Tₒₚₜ = {T_opt_pepc:.2f} K",
    )
    ax.set_title(title_pepc, fontsize=12, pad=6)
    ax.set_xlabel("Temperature (K)")
    ax.set_ylabel("Assimilation Rate (µmol m⁻² s⁻¹)")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.tick_params(axis="both", which="major", labelsize=10)
    ax.legend()

    # Rubisco plot
    ax = axes[1]
    ax.plot(T_range, Vcmax, color="darkorange", linewidth=2.5, label="Rubisco")
    ax.axvline(
        T_opt_rubisco,
        color="darkorange",
        linestyle="--",
        alpha=0.7,
        label=f"Tₒₚₜ = {T_opt_rubisco:.2f} K",
    )
    ax.set_title(title_rubisco, fontsize=12, pad=6)
    ax.set_xlabel("Temperature (K)")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.tick_params(axis="both", which="major", labelsize=10)
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Define external temperature range in Kelvin
T_range_K = np.linspace(283.15, 353.15, 200)  # 10°C to 50°C in Kelvin

# Generate the plots
plot_assimilation_subplots(T_range_K, param_dict)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def test_vpmax_temp_response(param_dict):
    R = param_dict["R"]
    T_range = np.linspace(260, 350, 200)

    vpmax_vals = []
    for T in T_range:
        exp_term = np.exp(param_dict["Ea_Vpmax"] * (T - 298.15) / (298.15 * R * T))
        deactivation = (
            1
            + np.exp(
                (298.15 * param_dict["DS_Vpmax"] - param_dict["Hd_Vpmax"])
                / (298.15 * R)
            )
        ) / (
            1 + np.exp((T * param_dict["DS_Vpmax"] - param_dict["Hd_Vpmax"]) / (T * R))
        )
        vpmax_T = param_dict["Vpmax_25"] * exp_term * deactivation
        vpmax_vals.append(vpmax_T)

    T_opt = T_range[np.argmax(vpmax_vals)]

    plt.plot(
        T_range,
        vpmax_vals,
        color="purple",
        linewidth=2,
        label="Vpmax Temperature Response",
    )
    plt.axvline(T_opt, color="red", linestyle="--", label=f"Optimum = {T_opt:.2f} K")
    plt.xlabel("T_mes [K]")
    plt.ylabel("Vpmax_T")
    plt.legend()
    plt.grid(True)
    plt.title("PEPC Vpmax vs Temperature")
    plt.show()

In [ ]:
test_vpmax_temp_response(param_dict)

## 🌱 Coupled Hydraulic–Temperature Model (Steady-State)

### 🔁 1. Water Fluxes (mol H₂O s⁻¹ m⁻²)

1. **Soil to Bundle Sheath**:
$$
F_{\text{soil-bs}} = g_{\text{soil}} \cdot (\psi_{\text{soil}} - \psi_{\text{bs}})
$$

2. **Bundle Sheath to Mesophyll**:
$$
F_{\text{bs-mes}} = g_{\text{bs}} \cdot (\psi_{\text{bs}} - \psi_{\text{mes}})
$$

3. **Transpiration from Mesophyll**:
$$
E_{\text{mes}} = g_s \cdot (\psi_{\text{mes}} - \psi_{\text{air}}) = g_s \cdot \frac{VPD}{P} = g_s \cdot \frac{e_s(T_{\text{mes}}) - e_a}{P}
$$

4. **Circular Flow (Cooling Loop)**:
$$
F = \text{const.}
$$

---

### 💧 2. Water Inputs and Losses

- **Rainfall**: $R$
- **Leakage**: $L = \lambda R$
- **Runoff**: $S = \sigma R$
- **Soil Evaporation**: $E_{\text{soil}}$

---

### 🔄 3. Water Balance Equations (Steady-State)

1. **Mesophyll**:
$$
F_{\text{bs-mes}} - E_{\text{mes}} = 0
$$

2. **Bundle Sheath**:
$$
F_{\text{soil-bs}} - F - F_{\text{bs-mes}} = 0
$$

3. **Soil**:
$$
R - \lambda R - \sigma R - E_{\text{soil}} + F = 0
\quad \Rightarrow \quad
F = (1 - \lambda - \sigma) R - E_{\text{soil}}
$$

---

### 🌡️ 4. Temperature (Energy) Balance Equations (Steady-State)

1. **Mesophyll**:
$$
A_{\text{mes}} \cdot \phi_{\text{mes}} 
- A_{\text{mes}} \cdot \gamma_{\text{mes}} T_{\text{mes}} 
- A_{\text{mes}} \cdot h_{\text{mes}} (T_{\text{mes}} - T_{\text{air}}) 
- U_{\text{mes-bs}} \cdot A_{\text{contact}} (T_{\text{mes}} - T_{\text{bs}}) 
- \kappa \cdot E_{\text{mes}} = 0
$$

2. **Bundle Sheath**:
$$
A_{\text{bs}} \cdot \phi_{\text{bs}} 
- A_{\text{bs}} \cdot \gamma_{\text{bs}} T_{\text{bs}} 
+ U_{\text{mes-bs}} \cdot A_{\text{contact}} (T_{\text{mes}} - T_{\text{bs}}) 
- C_w \cdot F \cdot A_{\text{circ}} (T_{\text{bs}} - T_{\text{soil}}) = 0
$$

3. **Soil**:
$$
\phi_{\text{soil}} 
- \gamma_{\text{soil}} T_{\text{soil}} 
- \beta_{\text{soil}} (T_{\text{soil}} - T_{\text{air}}) 
- \kappa \cdot E_{\text{soil}} 
+ C_w \cdot F \cdot A_{\text{circ}} (T_{\text{bs}} - T_{\text{soil}}) = 0
$$



# Summary of the model 

The model consist of 3 compartments: mesophyll, bundle sheath and soil. The water fluxes are calculated based on the potential difference between compartments and the hydraulic conductance of each compartment. The model  include the temperature balance equations for each compartment, which are couple to hydraulics via the circular flow of water ($F$). The opbjective function is to maximize the assimilation rates of PEPC and Rubisco, which are temperature dependent. The optimization problem becomes:

Given the environmental con ditions (solar irradiance, air temperature, and vapor pressure deficit), the model computes the optimal temperatures of the mesophyll and bundle sheath that maximize the assimilation rates while satisfying the energy balance equations. Furthr more the transpiration rate should be less or equal to  ($E_{mes} \leq g_{soil}\cdot \psi_{soil}-E_{soil}$).

The objective function is defined as:

## 🎯 Objective Function: C₄ Photosynthesis Optimization

This function maximizes Rubisco-limited carbon assimilation by adjusting:

- $g_s$: Stomatal conductance [mol m⁻² s⁻¹ Pa⁻¹]
- $F$: Circular water flow for cooling [kg m⁻² s⁻¹]

It accounts for:

- PEPC and Rubisco kinetics with temperature dependence
- Transpiration driven by VPD
- Coupling between hydraulics and leaf energy balance

---

### 🌬️ Transpiration Rate

Saturation vapor pressure using Tetens equation:
$$
e_s(T_{\text{air}}) = a \cdot \exp\left( \frac{b \cdot T}{T + c} \right)
$$

Vapor pressure deficit:
$$
VPD = e_s(T_{\text{air}}) \cdot (1 - RH)
$$

Effective conductance:
$$
g_{\text{air}} = \left( \frac{1}{g_s} + \frac{1}{g_m} \right)^{-1}
$$

Transpiration rate:
$$
E_{\text{mes}} = g_{\text{air}} \cdot VPD \cdot M_{\text{H2O}} \quad \text{[kg m⁻² s⁻¹]}
$$

---

### 🌡️ Leaf Energy Balance

Solves:
$$
(T_{\text{mes}}, T_{\text{bs}}, T_{\text{soil}}) = \text{calculate\_temperatures}(F, E_{\text{mes}})
$$
also considering the hydraulic equations


---

### 🍃 Rubisco-Limited Assimilation

Rubisco activity:
$$
V_{\text{cmax}}(T) = V_{\text{cmax,25}} \cdot \exp\left( \frac{E_a (T - T_0)}{RT_0 T} \right) \cdot \text{deactivation}
$$

Bundle sheath concentrations:
$$
[CO_2] = \frac{p_{CO_2}}{RT}, \quad [O_2] = \frac{p_{O_2}}{RT}
$$

Net CO₂ term:
$$
[CO_2]_{\text{net}} = [CO_2] - \frac{1}{2} \cdot \frac{[O_2]}{S_{c/o}}
$$

Rubisco-limited assimilation rate:
$$
A_{\text{rubisco}} = \frac{V_{\text{cmax}} \cdot [CO_2]_{\text{net}}}{[CO_2] + K_c \cdot \left(1 + \frac{[O_2]}{K_o} \right)}
$$

---

### 🔻 Objective to Minimize

Maximizing assimilation means minimizing:
$$
\text{Objective} = -A_{\text{rubisco}}
$$

---

### 🔒 Transpiration Upper Bound

Constraint (parameters only):
$$
E_{\text{mes}} \leq g_{\text{soil}} \cdot \psi_{\text{soil}} - E_{\text{soil}}
$$


## 📋 External Parameters for C₄ Photosynthesis–Hydraulics Model

### 🌡️ Energy & Heat Transfer
- $C_w = 4184$ J kg⁻¹ K⁻¹: Heat capacity of water
- $\kappa = 2.45 \times 10^6$ J kg⁻¹: Latent heat of vaporization
- $T_{\text{air}} = 308.15$ K, $T_{\text{sink}} = 283.15$ K
- $\gamma_{\text{mes}}, \gamma_{\text{bs}}, \gamma_{\text{soil}}$: Radiative loss coefficients
- $h_{\text{mes}}, h_{\text{soil}}$: Convective heat transfer
- $k_{\text{soil}} = 0.5$ W m⁻¹ K⁻¹, $d_{\text{soil}} = 0.5$ m
- $U_{\text{mes-bs}} = 50$, $U_{\text{surf-root}} = 25$, $U_{\text{root-sink}} = 20$ (all W m⁻² K⁻¹)

---

### 🔁 Water Transport (All in mol m⁻² s⁻¹)
- $E_{\text{surf}} \approx 1.11 \times 10^{-3}$ mol m⁻² s⁻¹
- $g_s \in [10^{-8}, 10^{-5}]$ mol m⁻² s⁻¹ Pa⁻¹: Stomatal conductance (optimization variable)
- $F \in [0, 0.111]$ mol m⁻² s⁻¹: Circular water return (converted from kg using $M_{\text{H2O}} = 0.01801528$ kg mol⁻¹)
- $g_{\text{soil}}, \psi_{\text{soil}}$: Required for transpiration constraint

---

### 🌞 Radiation Absorption
- $\phi = 1000$ W m⁻²: Incident light
- $\alpha_{\text{mes}} = 0.5$, $\alpha_{\text{bs}} = 0.6$, $\alpha_{\text{surf}} = 0.7$
- $\tau_{\text{mes}} = \tau_{\text{bs}} = 0.4$

---

### 🌿 Morphological Parameters
- $A_{\text{mes}} = 2.5$, $A_{\text{bs}} = 1.5$, $A_{\text{root}} = 1.0$, $A_{\text{contact}} = 0.5$
- $A_{\text{circ}} = 0.6$, $\zeta = 0.85$: Canopy coverage

---

### 💨 Environmental Conditions
- $RH = 0.8$, $P_{\text{atm}} = 101325$ Pa
- Tetens constants for $e_s$: $a = 0.6108$, $b = 17.27$, $c = 237.3$
- Air temperature function: $T_{\text{air}} = a \cdot \phi + b$

---

### 💧 Rainfall-Based Inputs
- $R \approx 1.28 \times 10^{-3}$ mol m⁻² s⁻¹: Rainfall rate (converted from 2.31e-5 kg m⁻² s⁻¹)
- $\lambda$, $\sigma$: Leakage and runoff fractions (implied)
- $E_{\text{bound}} = g_{\text{soil}} \cdot \psi_{\text{soil}} - E_{\text{soil}}$

---

### 🧬 Enzyme Kinetics (Rubisco + PEPC)
- Temperature response parameters: $V_{\text{cmax,25}}$, $E_a$, $DS$, $H_d$ for both Rubisco and PEPC
- $K_c$, $K_o$, $S_{c/o}$: Michaelis constants
- $k_{\text{cat,CA}} = 100$, $K_{\text{CA}} = 0.001$, $CA = 0.01$: Carbonic anhydrase parameters

---

### ⚖️ Objective Weights & Bounds
- Penalties: $\lambda_{\text{water}} = 10$, $\lambda_{\text{VPD}} = 5$, $\lambda_{\text{heat}} = 0.2$
- Target temperatures: $T_{\text{mes,opt}} = 313.15$ K, $T_{\text{bs,opt}} = 306.15$ K


